# 1. Preliminary Settings

First we need to take care of things like installations, Python imports, setting the random seed for the notebook and importing the csv files we need.

In [ ]:
# Installations

%pip install holidays
%pip install pandas numpy matplotlib seaborn
%pip install scikit-learn
%pip install "darts[torch,lightgbm]"
%pip install "darts[xgboost]"
%pip install --upgrade "scikit-learn<1.6.0"
%pip install statsforecast xgboost
%pip install lightgbm pyarrow
%pip install xgboost

In [ ]:
# Imports

import copy
import random
import sys
import warnings
import holidays
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import xgboost as xgb
from darts import TimeSeries, concatenate
from darts.metrics import mae, rmse
from darts.models import ARIMA, LightGBMModel, RNNModel, TFTModel, XGBModel
from pandas.plotting import autocorrelation_plot
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Setting the random seed for the entire notebook to ensure reproducibility

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # This ensures that even the internal GPU algorithms are deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("❄️ Global seeds frozen at 42")

In [ ]:
# # Import weather data from csv file

# # Import open-meteo weather data
# file_path = ADD PATH HERE
# open_meteo = pd.read_csv(file_path)

# # Display first rows and basic info
# display(open_meteo.head())
# open_meteo.info()

In [ ]:
# # Import sales data from csv file

# # - encoding: file uses Windows Latin-1 (cp1252) for German characters (Ä, ö, etc.)
# # - sep: columns are separated by semicolons (;)
# # - decimal: numbers use comma as decimal separator (e.g. 7,958)
# artikelbericht_path = ADD PATH HERE
# artikelbericht = pd.read_csv(
#     artikelbericht_path,
#     encoding="cp1252",
#     sep=";",
#     decimal=","
# )

# # Display first rows and basic info
# display(artikelbericht.head())
# artikelbericht.info()

In [ ]:
# Create "public holiday" DataFrame: every day 2020–2040 with weekday and Austrian public holiday flag

# All dates from 2020-01-01 to 2040-12-31
date_range = pd.date_range(start="2020-01-01", end="2040-12-31", freq="D")

# Austrian public holidays for the full range (covers moving holidays like Easter)
at_holidays = holidays.Austria(years=range(2020, 2041))

public_holiday = pd.DataFrame({
    "date": date_range,
    "weekday": date_range.day_name(),
    "public_holiday_flag": [
        "Yes" if d in at_holidays else "No"
        for d in date_range
    ]
})

# Optional: set date as index for easier merging later
# public_holiday = public_holiday.set_index("date")

display(public_holiday.head(10))
display(public_holiday[public_holiday["public_holiday_flag"] == "Yes"].head(20))
public_holiday.info()

# 2. Data Understanding

## 2.1. Data Understanding — Weather Data (source: open_meteo.csv)

Weather is a key external factor for supermarket demand according to some researchers (e.g. temperature, precipitation). We analyze the weather dataset from multiple angles in the following codeblocks.

In [ ]:
# Check Setup and temporal coverage

# Parse time as datetime (format DD/MM/YYYY)
open_meteo["time"] = pd.to_datetime(open_meteo["time"], format="%d/%m/%Y")
open_meteo = open_meteo.sort_values("time").reset_index(drop=True)

# Basic dataset characteristics
print("=== OPEN_METEO — Dataset overview ===\n")
print(f"Shape: {open_meteo.shape[0]} rows, {open_meteo.shape[1]} columns")
print(f"Date range: {open_meteo['time'].min()} to {open_meteo['time'].max()}")
print(f"Expected days (2020–2028): {(open_meteo['time'].max() - open_meteo['time'].min()).days + 1}")
print(f"Missing dates: {pd.date_range(open_meteo['time'].min(), open_meteo['time'].max()).difference(open_meteo['time'].to_list()).size}")
print("\nMissing values per column:")
print(open_meteo.isnull().sum()[open_meteo.isnull().sum() > 0] if open_meteo.isnull().sum().sum() > 0 else "None")
print("\nDtypes:")
print(open_meteo.dtypes)

In [ ]:
# Show univariate statistics for some selected columns to get a better feeling for the data
key_cols = [
    "temperature_2m_mean (°C)",
    "temperature_2m_max (°C)",
    "temperature_2m_min (°C)",
    "precipitation_sum (mm)",
    "relative_humidity_2m_mean (%)",
    "wind_speed_10m_max (km/h)",
    "cloud_cover_mean (%)",
]
print("=== OPEN_METEO — Descriptive statistics (key variables) ===\n")
display(open_meteo[key_cols].describe())
print("\nSkewness (relevant for forecasting assumptions):")
print(open_meteo[key_cols].skew())

In [ ]:
# Show time-series plots for some selected columns to get a better feeling for the data
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
x = open_meteo["time"]

axes[0].plot(x, open_meteo["temperature_2m_mean (°C)"], color="steelblue", linewidth=0.6, alpha=0.9)
axes[0].set_ylabel("Mean temp (°C)")
axes[0].set_title("Daily mean temperature (time series)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, open_meteo["precipitation_sum (mm)"], color="teal", linewidth=0.6, alpha=0.9)
axes[1].set_ylabel("Precipitation (mm)")
axes[1].set_title("Daily precipitation sum")
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, open_meteo["relative_humidity_2m_mean (%)"], color="gray", linewidth=0.6, alpha=0.8)
axes[2].set_ylabel("Humidity (%)")
axes[2].set_xlabel("Date")
axes[2].set_title("Daily mean relative humidity")
axes[2].grid(True, alpha=0.3)

plt.suptitle("open_meteo — Key weather variables over time", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check seasonality and year-on-year changes for temperature variable
open_meteo["year"] = open_meteo["time"].dt.year
open_meteo["month"] = open_meteo["time"].dt.month

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Temperature by month (boxplot)
sns.boxplot(data=open_meteo, x="month", y="temperature_2m_mean (°C)", ax=axes[0], palette="Blues_r")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Mean temperature (°C)")
axes[0].set_title("Seasonality: temperature by month")
axes[0].set_xticks(range(12))
axes[0].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])

# Mean temperature by year (annual trend)
yearly_temp = open_meteo.groupby("year")["temperature_2m_mean (°C)"].agg(["mean", "std", "min", "max"])
axes[1].plot(yearly_temp.index, yearly_temp["mean"], marker="o", color="steelblue", label="Mean")
axes[1].fill_between(yearly_temp.index, yearly_temp["min"], yearly_temp["max"], alpha=0.2, color="steelblue")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Temperature (°C)")
axes[1].set_title("Annual trend: mean daily temperature by year (min–max band)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Looking at distributions and correlations of selected weather variables

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

open_meteo["temperature_2m_mean (°C)"].hist(ax=axes[0,0], bins=40, color="steelblue", edgecolor="white")
axes[0,0].set_xlabel("Mean temperature (°C)")
axes[0,0].set_ylabel("Frequency")
axes[0,0].set_title("Distribution of daily mean temperature")

open_meteo["precipitation_sum (mm)"].hist(ax=axes[0,1], bins=50, color="teal", edgecolor="white")
axes[0,1].set_xlabel("Precipitation (mm)")
axes[0,1].set_ylabel("Frequency")
axes[0,1].set_title("Distribution of daily precipitation (right-skewed)")

# Correlation heatmap (numeric columns only)
num_cols = open_meteo.select_dtypes(include=["number"]).columns
corr = open_meteo[num_cols].corr()
# Focus on a subset if too many columns
if len(corr) > 12:
    key_for_corr = [c for c in key_cols if c in corr.columns][:10]
    corr = open_meteo[key_for_corr].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1,0], square=True)
axes[1,0].set_title("Correlation matrix (key weather variables)")

# Weather code (WMO) distribution — categorical view
open_meteo["weather_code (wmo code)"].value_counts().head(15).plot(kind="barh", ax=axes[1,1], color="slategray")
axes[1,1].set_xlabel("Count")
axes[1,1].set_title("Most frequent weather codes (WMO)")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier identification for selected variables

key_cols = [
    "temperature_2m_mean (°C)",
    "temperature_2m_max (°C)",
    "temperature_2m_min (°C)",
    "precipitation_sum (mm)",
    "relative_humidity_2m_mean (%)",
    "wind_speed_10m_max (km/h)",
    "cloud_cover_mean (%)",
]
outliers_iqr = {}
for col in key_cols:
    Q1 = open_meteo[col].quantile(0.25)
    Q3 = open_meteo[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    mask = (open_meteo[col] < lower) | (open_meteo[col] > upper)
    outliers_iqr[col] = mask.sum()
    print(f"{col}: {mask.sum()} outlier days (IQR method, bounds [{lower:.2f}, {upper:.2f}])")

fig, axes = plt.subplots(4, 2, figsize=(15, 15))
for ax, col in zip(axes.flat, key_cols):
    open_meteo.boxplot(column=col, ax=ax)
    ax.set_title(col)
plt.suptitle("open_meteo — Outlier identification (boxplots)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Checking autocorrelations of temperature and precipitation

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
temp = open_meteo.set_index("time")["temperature_2m_mean (°C)"]
autocorrelation_plot(temp, ax=axes[0])
axes[0].set_title("Autocorrelation of daily mean temperature")
axes[0].set_xlabel("Lag (days)")
# ACF for precipitation (often sparse)
precip = open_meteo.set_index("time")["precipitation_sum (mm)"]
autocorrelation_plot(precip, ax=axes[1])
axes[1].set_title("Autocorrelation of daily precipitation")
axes[1].set_xlabel("Lag (days)")
plt.tight_layout()
plt.show()

## 2.2. Data Understanding — Sales Data (source: artikelbericht.csv)

Sales data is the target domain for this research. Different aspects will be analyzed in the following codeblocks in order to understand better the data we have at hand.

In [ ]:
# Looking at some statistics to get a first high-level understanding of the data

artikelbericht["EX_DATE"] = pd.to_datetime(artikelbericht["EX_DATE"], format="%d.%m.%Y")
artikelbericht = artikelbericht.sort_values("EX_DATE").reset_index(drop=True)

print("=== ARTIKELBERICHT — Dataset overview ===\n")
print(f"Shape: {artikelbericht.shape[0]} rows (transaction/line level), {artikelbericht.shape[1]} columns")
print(f"Date range: {artikelbericht['EX_DATE'].min()} to {artikelbericht['EX_DATE'].max()}")
print(f"Unique dates: {artikelbericht['EX_DATE'].nunique()}")
print(f"Unique products (EX_PLU): {artikelbericht['EX_PLU'].nunique()}")
print(f"Unique departments (EX_DEPTEXT): {artikelbericht['EX_DEPTEXT'].nunique()}")
print("\nMissing values per column:")
miss = artikelbericht.isnull().sum()
print(miss[miss > 0] if miss.sum() > 0 else "None")
print("\nSample of key columns:")
display(artikelbericht[["EX_DATE", "EX_DEP", "EX_DEPTEXT", "EX_PLU", "EX_PLUTEXT", "EX_ITEM", "EX_WEIGHT", "EX_AMOUNT",]].head(10))

In [ ]:
# Some more statistical insights for selected variables

print("=== ARTIKELBERICHT — Descriptive statistics (numeric key variables) ===\n")
display(artikelbericht[["EX_NETTO", "EX_AMOUNT", "EX_ITEM", "EX_WEIGHT", "EX_PROFIT"]].describe())
print("\nSkewness:")
print(artikelbericht[["EX_NETTO", "EX_AMOUNT", "EX_ITEM", "EX_WEIGHT"]].skew())
print("\nDepartment mix (transaction count):")
display(artikelbericht["EX_DEPTEXT"].value_counts())

In [ ]:
# Looking at statistics for daily aggregations

daily = artikelbericht.groupby("EX_DATE").agg(
    daily_revenue=("EX_NETTO", "sum"),
    daily_quantity=("EX_ITEM", "sum"),
    unique_products=("EX_PLU", "nunique"),
).reset_index()

print("=== ARTIKELBERICHT — Daily aggregates ===\n")
display(daily.describe())
print(f"\nDays with sales: {len(daily)}")
plt.figure(figsize=(12, 4))
plt.plot(daily["EX_DATE"], daily["daily_revenue"], color="darkgreen", linewidth=0.7, alpha=0.9)
plt.ylabel("Daily revenue (EX_NETTO)")
plt.xlabel("Date")
plt.title("Daily revenue over time (target-relevant series)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Exploring seasonality and weekday effects
daily["weekday"] = daily["EX_DATE"].dt.day_name()
daily["month"] = daily["EX_DATE"].dt.month
daily["year"] = daily["EX_DATE"].dt.year

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
# Revenue by weekday
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_rev = daily.groupby("weekday", observed=True)["daily_revenue"].agg(["mean", "std"]).reindex(order)
weekday_rev.plot(kind="bar", y="mean", yerr="std", ax=axes[0], capsize=3, color="steelblue", legend=False)
axes[0].set_xticklabels(order, rotation=45, ha="right")
axes[0].set_ylabel("Mean daily revenue")
axes[0].set_title("Weekday effect on daily revenue")
axes[0].grid(True, alpha=0.3, axis="y")

# Revenue by month (seasonality)
monthly_rev = daily.groupby("month")["daily_revenue"].mean()
monthly_rev.plot(ax=axes[1], marker="o", color="darkgreen")
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
axes[1].set_ylabel("Mean daily revenue")
axes[1].set_xlabel("Month")
axes[1].set_title("Seasonality: mean daily revenue by month")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison of net revenue between different product categories and distribution of daily revenue

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
dept_revenue = artikelbericht.groupby("EX_DEPTEXT")["EX_NETTO"].sum().sort_values(ascending=True)
dept_revenue.plot(kind="barh", ax=axes[0], color="teal", alpha=0.85)
axes[0].set_xlabel("Total revenue (EX_NETTO)")
axes[0].set_title("Total revenue by product category (full period)")

# Distribution of daily revenue
daily["daily_revenue"].hist(ax=axes[1], bins=50, color="darkgreen", edgecolor="white")
axes[1].set_xlabel("Daily revenue")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of daily revenue")
plt.tight_layout()
plt.show()

In [ ]:
# Looking at outliers of selected variables
# Daily revenue outliers (IQR)
if "daily" not in dir() or "daily_revenue" not in daily.columns:
    daily = artikelbericht.groupby("EX_DATE").agg(
        daily_revenue=("EX_NETTO", "sum"),
        daily_quantity=("EX_ITEM", "sum"),
        daily_transactions=("EX_PLU", "count"),
        unique_products=("EX_PLU", "nunique"),
    ).reset_index()
Q1_d, Q3_d = daily["daily_revenue"].quantile(0.25), daily["daily_revenue"].quantile(0.75)
IQR_d = Q3_d - Q1_d
lower_d, upper_d = Q1_d - 1.5 * IQR_d, Q3_d + 1.5 * IQR_d
outliers_daily = ((daily["daily_revenue"] < lower_d) | (daily["daily_revenue"] > upper_d)).sum()
print(f"Daily revenue: {outliers_daily} outlier days (IQR bounds [{lower_d:.2f}, {upper_d:.2f}])")

# Transaction-level EX_NETTO outliers (histogram)
Q1_t, Q3_t = artikelbericht["EX_NETTO"].quantile(0.25), artikelbericht["EX_NETTO"].quantile(0.75)
IQR_t = Q3_t - Q1_t
lower_t, upper_t = Q1_t - 1.5 * IQR_t, Q3_t + 1.5 * IQR_t
outliers_txn = ((artikelbericht["EX_NETTO"] < lower_t) | (artikelbericht["EX_NETTO"] > upper_t)).sum()
print(f"EX_NETTO (per transaction): {outliers_txn} outlier rows (IQR bounds [{lower_t:.2f}, {upper_t:.2f}])")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
daily.boxplot(column="daily_revenue", ax=axes[0])
axes[0].set_title("Daily revenue (outliers)")
artikelbericht["EX_NETTO"].clip(upper=artikelbericht["EX_NETTO"].quantile(0.99)).hist(ax=axes[1], bins=50, edgecolor="white")
axes[1].set_xlabel("EX_NETTO (top 1% clipped for visibility)")
axes[1].set_title("Distribution of EX_NETTO")
plt.tight_layout()
plt.show()

In [ ]:
# Checkin autocorrelation of daily revenues

fig, ax = plt.subplots(figsize=(10, 4))
rev_series = daily.set_index("EX_DATE")["daily_revenue"]
autocorrelation_plot(rev_series, ax=ax)
ax.set_title("Autocorrelation of daily revenue")
ax.set_xlabel("Lag (days)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.3. Data Understanding — Calendric Data (source: engineered in Python)

Public holidays and weekdays are strong drivers of supermarket demand according to some researchers. We analyze the engineered dataframe from different angles to understand its content.

In [ ]:
# Dataframe overview and check balance/imbalance

print("=== PUBLIC_HOLIDAY — Dataset overview ===\n")
print(f"Shape: {public_holiday.shape[0]} rows, {public_holiday.shape[1]} columns")
print(f"Date range: {public_holiday['date'].min()} to {public_holiday['date'].max()}")
print(f"\nPublic holiday flag balance:")
print(public_holiday["public_holiday_flag"].value_counts())
print(f"\nProportion of public holiday days: {(public_holiday['public_holiday_flag'] == 'Yes').mean():.2%}")
print("\nWeekday distribution (all days):")
print(public_holiday["weekday"].value_counts().sort_index())

In [ ]:
# Looking at holidays per year and weekday of holidays

public_holiday["year"] = public_holiday["date"].dt.year
holidays_only = public_holiday[public_holiday["public_holiday_flag"] == "Yes"]

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
holidays_per_year = holidays_only.groupby("year").size()
holidays_per_year.plot(kind="bar", ax=axes[0], color="coral", edgecolor="white")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of public holidays")
axes[0].set_title("Austrian public holidays per year (2020–2040)")
axes[0].tick_params(axis="x", rotation=45)

holidays_only["weekday"].value_counts().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
).plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Weekday")
axes[1].set_ylabel("Count")
axes[1].set_title("On which weekdays do Austrian public holidays fall?")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison amount of holidays vs. non-holidays
fig, ax = plt.subplots(figsize=(6, 4))
public_holiday["public_holiday_flag"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"], edgecolor="white")
ax.set_xticklabels(["No", "Yes"], rotation=0)
ax.set_ylabel("Number of days")
ax.set_xlabel("Public holiday")
ax.set_title("Public holiday vs non-holiday days")
plt.tight_layout()
plt.show()


# 3. Data Preparation

## 3.1. Selecting the Product in Focus

This research is focusing on predicting strawberry sales, as for the specific supermarket in scope, strawberries show the greatest need for sophisticated planning and forecasting.

In [ ]:
# Product Focus: Strawberries

# 1. Filter: Exact match for 'Obst' AND partial match for 'Erdbeere'
df_strawberries = artikelbericht.loc[
    (artikelbericht["EX_DEPTEXT"] == "Obst") & 
    (artikelbericht["EX_PLUTEXT"].str.contains("Erdbeere", case=False, na=False))
]

# 2. Group and Describe
# This will give us count, mean, std, min, 25%, 50%, 75%, and max for all numeric columns
stats = df_strawberries.groupby("EX_PLUTEXT").describe()

# 3. Display
with pd.option_context('display.max_columns', None):
    display(stats)

In [ ]:
# Transforming all unit-values of strawberries to weight-values to ensure the possibility of comparison

# 1. Create a mask for rows where EX_ITEM is not 0
mask = df_strawberries["EX_ITEM"] != 0

# 2. Multiply the value by 0.5 and move it to EX_WEIGHT
df_strawberries.loc[mask, "EX_WEIGHT"] = df_strawberries.loc[mask, "EX_ITEM"] * 0.5

# 3. Set the original EX_ITEM values to 0 for those rows
df_strawberries.loc[mask, "EX_ITEM"] = 0

# Quick verification
print(f"Items processed: {mask.sum()}")
display(df_strawberries.loc[mask, ["EX_ITEM", "EX_WEIGHT"]].head())

In [ ]:
# Check the characteristics of the df again to make sure all item values were translated to weight values

df_strawberries.groupby("EX_PLUTEXT").describe()

In [ ]:
# 1. Create the mask to find all variations (Tasse, 2. Wahl, etc.)

# Using "Erdbeer" captures both singular and plural forms
strawberry_mask = df_strawberries["EX_PLUTEXT"].str.contains("Erdbeer", case=False, na=False)

# 2. Unify the text label to "Strawberries"
df_strawberries.loc[strawberry_mask, "EX_PLUTEXT"] = "Erdbeeren"

# Change the entire column to the integer 10004
df_strawberries['EX_PLU'] = 10004

# 4. Quick check to see the unification
print(f"Successfully updated {strawberry_mask.sum()} rows in df_strawberries.")
display(df_strawberries[["EX_PLU", "EX_PLUTEXT"]].drop_duplicates())

In [ ]:
# Look at the newly created df to verify correctness

df_strawberries.head(100)

## 3.2. Merging and Cleaning the Datasets

In this step, sales, weather and calendric data are merged into one dataframe, so this df can be the foundation for all further steps.

In [ ]:
# Create the final unified dataframe starting with weather data as the base

# We use a left join also for the df_starwberries, as we also want to have a row for each day on which no strawberries were sold
df_strawberries_merged = open_meteo.merge(
    public_holiday,
    left_on="time",
    right_on="date",
    how="left"
).merge(
    df_strawberries,
    left_on="time",
    right_on="EX_DATE",
    how="left"
)

# Print results to verify the merge
print(f"Final dataframe rows: {df_strawberries_merged.shape[0]}")
with pd.option_context('display.max_columns', None):
    display(df_strawberries_merged.head())

In [ ]:
# Filter to keep only rows where the year is between 2021 and 2025 inclusive, as no strawberries were sold before 2021
df_strawberries_merged = df_strawberries_merged[df_strawberries_merged['time'].dt.year.between(2021, 2025)]

# Verification: Check the unique years remaining
print(f"Years remaining in dataset: {sorted(df_strawberries_merged['time'].dt.year.unique())}")
print(f"Final row count: {len(df_strawberries_merged)}")

In [ ]:
# Remove columns which will certainly not be needed for the modeling phase

# List of columns to be removed
columns_to_drop = [
    "year_x", "month", "date", "year_y", "EX_BRANCH", "EX_CASHREG", 
    "EX_CLIENT", "EX_CLIENTGROUP", "EX_DATE", "EX_STORE", "EX_DEP", 
    "EX_DEPTEXT", "EX_PLU", "EX_ITEM", "EX_TAXCODE", "EX_PROMOTION", 
    "EX_MAJGROUP", "EX_MAJGROUPTEXT", "EX_MINGROUP", "EX_MINGROUPTEXT"
]

# Drop the columns from the dataframe
df_strawberries_merged = df_strawberries_merged.drop(columns=columns_to_drop, errors='ignore')

# Verification: Show the remaining columns
print(f"Remaining columns: {df_strawberries_merged.columns.tolist()}")
display(df_strawberries_merged.head())

In [ ]:
# Fill NaN values, that are arising due to the left join (on days where no strawberries were sold). They need to be filled in, as many algorithms can't handle NaN values

# 1. Fill the text column with "Erdbeeren"
df_strawberries_merged["EX_PLUTEXT"] = df_strawberries_merged["EX_PLUTEXT"].fillna("Erdbeeren")

# 2. Identify all other columns starting with 'EX_'
# We exclude 'EX_PLUTEXT' since we just handled it
ex_cols_to_zero = [col for col in df_strawberries_merged.columns 
                   if col.startswith("EX_") and col != "EX_PLUTEXT"]

# 3. Fill those columns with 0.0 (float)
df_strawberries_merged[ex_cols_to_zero] = df_strawberries_merged[ex_cols_to_zero].fillna(0.0)

# Verification: Check for any remaining NaNs in EX_ columns
print(df_strawberries_merged[[col for col in df_strawberries_merged.columns if col.startswith("EX_")]].isna().sum())
display(df_strawberries_merged.head())

In [ ]:
# To get a better feeling for the specific strawberry sales data, we plot the sales (in kg) per day over time. 
# This should also help understand, if there were some obvious problems during the preprocessing of the dataset.

# 1. Aggregate weight by date to get the total daily volume
daily_sales = df_strawberries_merged.groupby('time')['EX_WEIGHT'].sum().reset_index()

# 2. Create a wider plot using subplots
# figsize=(width, height) - we increase width to 18 for a better view
fig, ax = plt.subplots(figsize=(18, 6))

# 3. Plotting the data
ax.plot(daily_sales['time'], daily_sales['EX_WEIGHT'], linewidth=1, label='Daily Weight')

# 4. Formatting for clarity
ax.set_title('Daily Distribution of EX_WEIGHT (2021 - 2025)', fontsize=16)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Weight', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)

# Ensure the x-axis labels are readable
plt.xticks(rotation=45)

# 5. Save the high-resolution wider plot
plt.tight_layout()
plt.savefig('strawberry_weight_wide.png')

In [ ]:
# As zero-sales during the peak season (May-October) indicates stockout, we need to impute values into these zero-sales days, in order to provide the algorithms with realistic demand data.

# Identify Stockouts (May-Oct where Weight is 0)
is_peak = df_strawberries_merged['time'].dt.month.between(5, 10)
is_zero = (df_strawberries_merged['EX_WEIGHT'] == 0)
df_strawberries_merged['is_stockout'] = (is_peak & is_zero).astype(int)

# Store original weight
df_strawberries_merged['EX_WEIGHT_ORIGINAL'] = df_strawberries_merged['EX_WEIGHT'].copy()

# 2. Impute Row-by-Row to prevent Data Leakage
def leakage_free_impute(row, df):
    if row['is_stockout'] == 0:
        return row['EX_WEIGHT']
    
    current_time = row['time']
    
    # Get all DATA from the PAST (Strictly before the current date)
    past_data = df[(df['time'] < current_time) & (df['EX_WEIGHT_ORIGINAL'] > 0)]
    
    # OPTION A: Seasonal Historical Average (Same month, Same day of week)
    if not past_data.empty:
        seasonal_match = past_data[
            (past_data['time'].dt.month == current_time.month) & 
            (past_data['time'].dt.dayofweek == current_time.dayofweek)
        ]
        
        if not seasonal_match.empty:
            return seasonal_match['EX_WEIGHT_ORIGINAL'].mean()
        
        # OPTION B: Fallback - Recent Rolling Mean (Last 14 days of history)
        recent_data = past_data.tail(14)
        if not recent_data.empty:
            return recent_data['EX_WEIGHT_ORIGINAL'].mean()
            
    # OPTION C: If literally no past data exists (e.g., start of 2021)
    return 0 

# Apply the function
# Note: On very large datasets, this can be slow. If it takes too long, 
# we can optimize with a vectorized 'groupby' on past years only.
df_strawberries_merged['EX_WEIGHT'] = df_strawberries_merged.apply(
    lambda x: leakage_free_impute(x, df_strawberries_merged), axis=1
)

print(f"Imputation complete. Stockout flag maintained for {df_strawberries_merged['is_stockout'].sum()} rows.")

In [ ]:
# Look at the graph again to see what changed after the imputation

# 1. Aggregate weight by date to get the total daily volume
daily_sales = df_strawberries_merged.groupby('time')['EX_WEIGHT'].sum().reset_index()

# 2. Create a wider plot using subplots
# figsize=(width, height) - we increase width to 18 for a better view
fig, ax = plt.subplots(figsize=(18, 6))

# 3. Plotting the data
ax.plot(daily_sales['time'], daily_sales['EX_WEIGHT'], linewidth=1, label='Daily Weight')

# 4. Formatting for clarity
ax.set_title('Daily Distribution of EX_WEIGHT (2021 - 2025)', fontsize=16)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Weight', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)

# Ensure the x-axis labels are readable
plt.xticks(rotation=45)

# 5. Save the high-resolution wider plot
plt.tight_layout()
plt.savefig('strawberry_weight_wide.png')

In [ ]:
# Preparing a dataframe that only contains numeric values (plus one date column)

# List of specific columns to be removed
additional_cols_to_drop = ["weather_code (wmo code)", "sunrise (iso8601)", "sunset (iso8601)", "EX_PLUTEXT"]

# Drop the columns from the dataframe
df_strawberries_merged = df_strawberries_merged.drop(columns=additional_cols_to_drop, errors='ignore')

# Verification: Show the final set of columns
print(f"Columns after cleanup: {df_strawberries_merged.columns.tolist()}")
display(df_strawberries_merged.head())

In [ ]:
# Transform the remaining string values - creating a 0/1 boolean value for public_holiday_flag

# Map 'Yes' to 1 and 'No' to 0
# We use .fillna(0) to ensure any missing values are treated as non-holidays
df_strawberries_merged['public_holiday_flag'] = df_strawberries_merged['public_holiday_flag'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

# Verification: Show the distribution of the new numeric flag
print("Value counts for public_holiday_flag:")
print(df_strawberries_merged['public_holiday_flag'].value_counts())

# Display the first few rows to see the change
display(df_strawberries_merged[['time', 'public_holiday_flag']].head())

In [ ]:
# To prepare the dataset for Train-Validation-Test Split, we change the column name "time" to "date" in order to avoid confusion

# Rename 'time' to 'date'
df_strawberries_merged = df_strawberries_merged.rename(columns={'time': 'date'})

# Verification
print(f"New column name: {df_strawberries_merged.columns[0]}") # Assumes it's the first column
display(df_strawberries_merged.head())

In [ ]:
# Cyclical Encoding of the weekday column to ensure that there are no string values left in the dataset

import numpy as np

# 1. Map the string names to numbers 0-6
weekday_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 
    'Friday': 4, 'Saturday': 5, 'Sunday': 6
}
df_strawberries_merged['weekday_num'] = df_strawberries_merged['weekday'].map(weekday_map)

# 2. Apply Sine and Cosine transformations (7-day cycle)
df_strawberries_merged['weekday_sin'] = np.sin(2 * np.pi * df_strawberries_merged['weekday_num'] / 7)
df_strawberries_merged['weekday_cos'] = np.cos(2 * np.pi * df_strawberries_merged['weekday_num'] / 7)

# 3. Drop the original string column and the intermediate numeric column
df_strawberries_merged = df_strawberries_merged.drop(columns=['weekday', 'weekday_num'])

# Verification
display(df_strawberries_merged[['date', 'weekday_sin', 'weekday_cos']].head())

In [ ]:
# Also encode a month column by clcycal encoding

# 1. Extract the month from your new 'date' column
df_strawberries_merged['month'] = df_strawberries_merged['date'].dt.month

# 2. Apply Cyclical Encoding (Sine and Cosine)
# We divide by 12 because there are 12 months in a year
df_strawberries_merged['month_sin'] = np.sin(2 * np.pi * df_strawberries_merged['month'] / 12)
df_strawberries_merged['month_cos'] = np.cos(2 * np.pi * df_strawberries_merged['month'] / 12)

# 3. Optional: Drop the original month column if you only want the cyclical features
df_strawberries_merged = df_strawberries_merged.drop(columns=['month'])

display(df_strawberries_merged[['date', 'month_sin', 'month_cos']].head(100))

In [ ]:
# As we have seen a few codeblocks before, we have on average more than 1 row per day, due to different types of strawberries. This needs to be merged.

# 1. Identify which columns to sum and which to average
# We sum everything starting with 'EX_'
# We take the mean for everything else (Weather, Lags, cyclical features)
cols_to_sum = [c for c in df_strawberries_merged.columns if c.startswith('EX_')]
cols_to_keep = [c for c in df_strawberries_merged.columns if c not in cols_to_sum and c != 'date']

# 2. Define the aggregation dictionary
# This tells pandas exactly what math to do for every column
agg_dict = {col: 'sum' for col in cols_to_sum}
for col in cols_to_keep:
    agg_dict[col] = 'mean'

# 3. Group by 'date' and apply the aggregation
df_daily = df_strawberries_merged.groupby('date').agg(agg_dict).reset_index()

# 4. Sort and verify the results
df_daily = df_daily.sort_values('date')

print(f"Original Row Count: {len(df_strawberries_merged)}")
print(f"New '1 Row per Day' Count: {len(df_daily)}")

In [ ]:
# Check a date that was duplicated in the df before

display(df_daily[df_daily["date"] == "2024-05-22"])

In [ ]:
# Check the same date again in the older df - it shows the duplication

display(df_strawberries_merged[df_strawberries_merged["date"] == "2024-05-22"])

In [ ]:
# Calculate lags of 7 days and add them as features to the df, as some algorithms need this information to perform the prediction

df_strawberries_merged = df_daily
# 1. Sort by date to ensure shifts align correctly
df_strawberries_merged = df_strawberries_merged.sort_values('date')

# 2. Create the "Safe" 7-day Lags
# These are available for every day in your 1-7 day forecast window
df_strawberries_merged['lag_7'] = df_strawberries_merged['EX_WEIGHT'].shift(7)
df_strawberries_merged['lag_14'] = df_strawberries_merged['EX_WEIGHT'].shift(14)

# 3. Create a 7-day Rolling Mean (The average of the week ending TODAY)
# We shift by 7 so it's usable for the entire next week
df_strawberries_merged['rolling_mean_7'] = (
    df_strawberries_merged['EX_WEIGHT']
    .shift(7)
    .rolling(window=7)
    .mean()
)

# 4. Cleanup: Drop rows where we don't have enough history to calculate lags
df_strawberries_merged = df_strawberries_merged.dropna(subset=['lag_14', 'rolling_mean_7'])

display(df_strawberries_merged.head())

In [ ]:
# Create Pearson Correlation Matrix to reduce further the number of features

# 1. Select only numeric columns
# Pearson correlation requires numeric data; this ignores the 'date' column
numeric_df = df_strawberries_merged.select_dtypes(include=[np.number])

# 2. Compute the Pearson correlation matrix
corr_matrix = numeric_df.corr(method='pearson')

# 3. Setup the visualization
# We use a large figsize to ensure all labels are readable
fig, ax = plt.subplots(figsize=(18, 14))

# Create a mask to hide the upper triangle (it's a mirror of the lower one)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 4. Generate the heatmap
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True,          # Display the numeric values in the cells
    fmt=".2f",           # Limit to 2 decimal places
    cmap='coolwarm',     # Red for positive, Blue for negative correlation
    center=0,            # 0 is the neutral point (white)
    linewidths=.5, 
    square=True,
    cbar_kws={"shrink": .8}
)

plt.title('Pearson Correlation Matrix for Strawberry Sales Features', fontsize=18)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# 5. Save the plot
plt.tight_layout()
plt.savefig('correlation_matrix_strawberries.png')

# 6. Print highly correlated pairs (> 0.85)
# This helps you pinpoint exactly which columns to drop without squinting at the map
print("Highly correlated features (> 0.85):")
sol = (corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
                  .stack()
                  .sort_values(ascending=False))
for (f1, f2), val in sol.items():
    if abs(val) > 0.85:
        print(f"{f1} and {f2}: {val:.2f}")

In [ ]:
# Based on the results of the Pearson Correlation Matrix, some additional columns need to be dropped (due to very high correlations)

# List of highly correlated/redundant columns
cols_to_drop = [
    # Temperature Redundancies
    "temperature_2m_max (°C)", "temperature_2m_min (°C)", 
    "apparent_temperature_mean (°C)", "apparent_temperature_max (°C)", 
    "apparent_temperature_min (°C)", "shortwave_radiation_sum (MJ/m²)", "daylight_duration (s)",
    
    # Precipitation & Wind Redundancies
    "rain_sum (mm)", "snowfall_water_equivalent_sum (mm)",
    "wind_speed_10m_max (km/h)", "wind_gusts_10m_max (km/h)", 
    "wind_direction_10m_dominant (°)", "wind_speed_10m_mean (km/h)", "et0_fao_evapotranspiration (mm)",
    
    # Financial/Target Redundancies (to prevent target leakage)
    "EX_AMOUNT", "EX_NETTO", "EX_TAX", "EX_SUPPAMOUNT", 
    "EX_PROFIT", "EX_PROFITRATE", "EX_WEIGHT_ORIGINAL"
]

# Drop the columns from the dataframe
df_strawberries_merged_clean = df_strawberries_merged.drop(columns=cols_to_drop, errors='ignore')

# Verification: Show what is left
print(f"Remaining columns: {df_strawberries_merged_clean.columns.tolist()}")

In [ ]:
# Look at the correlation matrix without the dropped columns to see the difference

# 1. Select only numeric columns
# Pearson correlation requires numeric data; this ignores the 'date' column
numeric_df = df_strawberries_merged_clean.select_dtypes(include=[np.number])

# 2. Compute the Pearson correlation matrix
corr_matrix = numeric_df.corr(method='pearson')

# 3. Setup the visualization
# We use a large figsize to ensure all labels are readable
fig, ax = plt.subplots(figsize=(18, 14))

# Create a mask to hide the upper triangle (it's a mirror of the lower one)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 4. Generate the heatmap
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True,          # Display the numeric values in the cells
    fmt=".2f",           # Limit to 2 decimal places
    cmap='coolwarm',     # Red for positive, Blue for negative correlation
    center=0,            # 0 is the neutral point (white)
    linewidths=.5, 
    square=True,
    cbar_kws={"shrink": .8}
)

plt.title('Pearson Correlation Matrix for Strawberry Sales Features', fontsize=18)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# 5. Save the plot
plt.tight_layout()
plt.savefig('correlation_matrix_strawberries.png')

# 6. Print highly correlated pairs (> 0.85)
# This helps you pinpoint exactly which columns to drop without squinting at the map
print("Highly correlated features (> 0.85):")
sol = (corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
                  .stack()
                  .sort_values(ascending=False))
for (f1, f2), val in sol.items():
    if abs(val) > 0.85:
        print(f"{f1} and {f2}: {val:.2f}")

In [ ]:
# Display the df without the highly correlated features

display(df_strawberries_merged_clean)

In [ ]:
# Check the rows that are in peak season, but nevertheless have at least one of their lag features valued with zero

# Define the Peak Season mask
is_peak = df_strawberries_merged_clean['date'].dt.month.between(5, 10)

# Check for zeros in these columns during Peak Season
# We check if any of the three columns are 0 while in peak season
check_cols = ['lag_7', 'lag_14', 'rolling_mean_7']
zero_lags = df_strawberries_merged_clean[is_peak & (df_strawberries_merged_clean[check_cols] == 0).any(axis=1)]

if zero_lags.empty:
    print("✅ Success: No zero-values found in lags/rolling means during peak season!")
else:
    print(f"⚠️ Warning: Found {len(zero_lags)} rows where lags are still 0.")
    with pd.option_context('display.max_rows', None):
        display(zero_lags[['date', 'EX_WEIGHT'] + check_cols])

## 3.3. Train-Validation-Test Split

Up until here all the data preparation, that is needed and could be done before the split without causing data leakage is completed. Now we need to split the dataset into train, validation and test data (60-20-20 ratio; chronological), in order to proceed.

In [ ]:
# Perform the split

# 1. Ensure the 'date' column is datetime and sorted
df_strawberries_merged_clean['date'] = pd.to_datetime(df_strawberries_merged_clean['date'])
df_strawberries_merged_clean = df_strawberries_merged_clean.sort_values('date')

# 2. Define the Cutoff Points
# Train: 2021, 2022, 2023
# Validation: 2024
# Test: 2025
train_end = '2023-12-31'
val_end = '2024-12-31'

# 3. Create the Splits
train_df = df_strawberries_merged_clean[df_strawberries_merged_clean['date'] <= train_end].copy()
val_df = df_strawberries_merged_clean[(df_strawberries_merged_clean['date'] > train_end) & 
                                      (df_strawberries_merged_clean['date'] <= val_end)].copy()
test_df = df_strawberries_merged_clean[df_strawberries_merged_clean['date'] > val_end].copy()

# 4. Separate Features (X) and Target (y)
target = 'EX_WEIGHT'

cols_to_drop = [target, 'date']

X_train = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
y_train = train_df[target]

X_val = val_df.drop(columns=[c for c in cols_to_drop if c in val_df.columns])
y_val = val_df[target]

X_test = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])
y_test = test_df[target]

# 5. Print confirmation
print(f"Total Rows: {len(df_strawberries_merged_clean)}")
print(f"Training:   {len(X_train)} rows (Ends {train_df['date'].max().date()})")
print(f"Validation: {len(X_val)} rows (Year {val_df['date'].dt.year.unique()[0]})")
print(f"Testing:    {len(X_test)} rows (Year {test_df['date'].dt.year.unique()[0]})")

In [ ]:
# Display both dfs of the train set

display(X_train)
display(y_train)

## 3.4. Outlier Treatment

Now we need to decide what to do with the outliers, as it is crucial to perform this step only after the train-validation-test split.

In [ ]:
# Performing the IQR check on the prepared df

# 1. Statistical Check
Q1 = train_df['EX_WEIGHT'].quantile(0.25)
Q3 = train_df['EX_WEIGHT'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + (1.5 * IQR)

outliers = train_df[train_df['EX_WEIGHT'] > upper_limit]

print(f"IQR: {IQR:.2f}")
print(f"Upper Limit for Outliers: {upper_limit:.2f}")
print(f"Number of Outliers detected: {len(outliers)} ({len(outliers)/len(train_df)*100:.2f}%)")

# 2. Visual Check
plt.figure(figsize=(10, 4))
sns.boxplot(x=train_df['EX_WEIGHT'], color='skyblue')
plt.title('Strawberry Sales Outlier Detection (Training Set)')
plt.show()

# 3. Time-Series Check (Crucial!)
# Plotting to see if outliers are just 'Peak Season' or true errors
plt.figure(figsize=(15, 5))
plt.plot(train_df['date'], train_df['EX_WEIGHT'], label='Sales')
plt.axhline(y=upper_limit, color='r', linestyle='--', label='Outlier Threshold')
plt.title('Sales vs Outlier Threshold over Time')
plt.legend()
plt.show()

Conclusion: Treating the outliers (e.g. clipping) based on the IQR method would result in a major mistake, as the spikes of the high season would get lost. Even the extreme spikes in the high season of 2023 are important information, as it shows the general development of the supermarket, gaining popularity and increasing sales over the years.
Therefore the decision is made to not clip any values and instead keep them as they are to preserve the information concerning seasonality.

## 3.5. Scaling

As the features we are using have different magnitudes, we need to scale them to ensure the models will attribute them equal importance.

In [ ]:
# Perform scaling of the X_ dataframes

# 1. Initialize the Scaler
scaler = MinMaxScaler()

# 2. Fit on Training Data ONLY to prevent leakage
# We only scale the features (X), not the target (y) for now 
# (unless you specifically want to scale the target for the LSTM)
scaler.fit(X_train)

# 3. Transform all sets
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling complete. All features are now between 0 and 1.")

In [ ]:
# As some of the algorithms intended to train perform better if the target variable is also scaled, this code is dedicated to perform this scaling

# 1. Initialize a separate scaler for the target (y)
y_scaler = MinMaxScaler()

# 2. Reshape is required because the scaler expects a 2D array [[val1], [val2]]
y_train_reshaped = y_train.values.reshape(-1, 1)

# 3. Fit on Training set only
y_scaler.fit(y_train_reshaped)

# 4. Transform all sets for the LSTM
y_train_scaled = y_scaler.transform(y_train_reshaped)
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

# Useful at a later stage: the code for the inverse scaler
# predicted_kg = y_scaler.inverse_transform(model_predictions)

In [ ]:
# Checking the scaled df to see if everything went as expected

# 1. Identify the indices for June 2023 from the original train_df
june_2023_indices = train_df[(train_df['date'] >= '2023-06-01') & (train_df['date'] <= '2023-06-30')].index

# 2. Display the scaled features for those specific rows
print("--- Displaying X_train_scaled for June 2023 ---")
display(X_train_scaled.loc[june_2023_indices])

## 3.6. Splitting into Feature Subsets

One of the research objectives of this work is to understand which feature combination leads to the best forecasting results. Therefore we need to create different feature subsets at this point, so we can later on reach this objective.

In [ ]:
# As the goal is to identify which features are the most important for prediction, the dataframes are split into thematical subsets

# 1. Define the column groups based on your screenshot
sales_cols = ['lag_7', 'lag_14', 'rolling_mean_7', 'is_stockout']

weather_cols = [
    'temperature_2m_mean (°C)', 'sunshine_duration (s)', 'precipitation_sum (mm)', 
    'snowfall_sum (cm)', 'precipitation_hours (h)', 'cloud_cover_mean (%)', 
    'relative_humidity_2m_mean (%)', 'pressure_msl_mean (hPa)', 'wind_gusts_10m_mean (km/h)'
]

calendric_cols = [
    'public_holiday_flag', 'weekday_sin', 'weekday_cos', 
    'month_sin', 'month_cos'
]

# 2. Create the 4 Feature Subsets: s = sales, w = weather, c = calendar
# Subset 1: Only Sales
X_train_s = X_train_scaled[sales_cols]
X_val_s   = X_val_scaled[sales_cols]
X_test_s  = X_test_scaled[sales_cols]

# Subset 2: Sales + Weather
X_train_s_w = X_train_scaled[sales_cols + weather_cols]
X_val_s_w   = X_val_scaled[sales_cols + weather_cols]
X_test_s_w  = X_test_scaled[sales_cols + weather_cols]

# Subset 3: Sales + Calendric
X_train_s_c = X_train_scaled[sales_cols + calendric_cols]
X_val_s_c   = X_val_scaled[sales_cols + calendric_cols]
X_test_s_c  = X_test_scaled[sales_cols + calendric_cols]

# Subset 4: All (Already exists as your scaled dataframes)
X_train_s_w_c = X_train_scaled
X_val_s_w_c   = X_val_scaled
X_test_s_w_c  = X_test_scaled

print(f"Subsets created successfully!")
print(f"V1 (Sales Only): {X_train_s.shape[1]} features")
print(f"V2 (Sales + Weather): {X_train_s_w.shape[1]} features")
print(f"V3 (Sales + Calendric): {X_train_s_c.shape[1]} features")
print(f"V4 (Full Set): {X_train_s_w_c.shape[1]} features")

## 3.7. Preparing Datasets for Darts Library

The Darts library was chosen for this research, as it is one of the most popular libraries for time-series forecasting and it features implementations of all the algorithms that are in scope of this research.

In [ ]:
# Create the dataframes in the right format, so Darts can work with them

# 1. Helper Function with updated method name
def to_darts_ts(df_scaled, original_df):
    """Re-attaches dates and converts to Darts TimeSeries using from_dataframe."""
    temp = df_scaled.copy()
    temp['date'] = original_df['date'].values
    return TimeSeries.from_dataframe(temp, time_col='date')

# 2. Transform the Targets (y)
# --- Unscaled Targets (for ARIMA, LightGBM) ---
y_train_ts = TimeSeries.from_dataframe(train_df, time_col='date', value_cols='EX_WEIGHT')
y_val_ts   = TimeSeries.from_dataframe(val_df,   time_col='date', value_cols='EX_WEIGHT')
y_test_ts  = TimeSeries.from_dataframe(test_df,  time_col='date', value_cols='EX_WEIGHT')

# --- Scaled Targets (for LSTM, TFT) ---
y_train_scaled_ts = TimeSeries.from_dataframe(pd.DataFrame(y_train_scaled, columns=['EX_WEIGHT'], index=train_df.index).assign(date=train_df['date']), time_col='date', value_cols='EX_WEIGHT')
y_val_scaled_ts   = TimeSeries.from_dataframe(pd.DataFrame(y_val_scaled,   columns=['EX_WEIGHT'], index=val_df.index).assign(date=val_df['date']),     time_col='date', value_cols='EX_WEIGHT')
y_test_scaled_ts  = TimeSeries.from_dataframe(pd.DataFrame(y_test_scaled,  columns=['EX_WEIGHT'], index=test_df.index).assign(date=test_df['date']),   time_col='date', value_cols='EX_WEIGHT')

# 3. Transform Feature Subsets (X)
# --- V1: Sales Only (_s) ---
X_train_s_ts = to_darts_ts(X_train_s, train_df)
X_val_s_ts   = to_darts_ts(X_val_s,   val_df)
X_test_s_ts  = to_darts_ts(X_test_s,  test_df)

# --- V2: Sales + Weather (_s_w) ---
X_train_s_w_ts = to_darts_ts(X_train_s_w, train_df)
X_val_s_w_ts   = to_darts_ts(X_val_s_w,   val_df)
X_test_s_w_ts  = to_darts_ts(X_test_s_w,  test_df)

# --- V3: Sales + Calendric (_s_c) ---
X_train_s_c_ts = to_darts_ts(X_train_s_c, train_df)
X_val_s_c_ts   = to_darts_ts(X_val_s_c,   val_df)
X_test_s_c_ts  = to_darts_ts(X_test_s_c,  test_df)

# --- V4: Full Set (_s_w_c) ---
X_train_s_w_c_ts = to_darts_ts(X_train_s_w_c, train_df)
X_val_s_w_c_ts   = to_darts_ts(X_val_s_w_c,   val_df)
X_test_s_w_c_ts  = to_darts_ts(X_test_s_w_c,  test_df)

print("✅ All TimeSeries objects created successfully!")

In [ ]:
# Checking a X_ dataframe to make sure the transformation worked correctly

display(X_train_s_ts)

In [ ]:
# Checking a y_ dataframe to make sure the transformation worked correctly

display(y_train_scaled_ts)

# 4. Modeling: Evaluation of Feature Subsets

Now all the initial preparation of the data is completed and we can start to actually train the different algrithms and consequently create the models. We will cover different algorithm families, start with a naive baseline and stepwise increase the complexity of the algorithms, up to training a temporal fusion transformer (TFT).

## 4.1. Baseline Model: Average Sales Amount

For being able to judge the performance of the later developed models, it's important to get a naive baseline as a benchmark that needs to be surpassed.

In [ ]:
# Develop the naive baseline

# 1. Define active months (April to October)
active_months = [4, 5, 6, 7, 8, 9, 10]

# 2. Extract months from the original Darts TimeSeries indices
# We use .time_index to get the actual timestamps
train_months = y_train_ts.time_index.month
val_months = y_val_ts.time_index.month

# 3. Calculate the average sales for active months in training data
# Create a mask using the extracted month sequence
train_active_mask = np.isin(train_months, active_months)

# Calculate mean from y_train DataFrame using the mask
train_mean_active = y_train.iloc[train_active_mask].values.mean()

print(f"📊 Training Mean (April-Oct): {train_mean_active:.3f} kg")

# 4. Create the Naive Forecast for the validation period
# Initialize zeros with the same shape as y_val
naive_preds = np.zeros(y_val.shape)

# Identify where the validation set falls in April-Oct
val_active_mask = np.isin(val_months, active_months)

# Assign the mean to those specific slots
naive_preds[val_active_mask] = train_mean_active

# 5. Calculate Metrics
actual = y_val.values.flatten()
forecast = naive_preds.flatten()

# MAE: Mean Absolute Error
res_mae = np.mean(np.abs(actual - forecast))

# RMSE: Root Mean Squared Error
res_rmse = np.sqrt(np.mean((actual - forecast)**2))

# WAPE: Weighted Absolute Percentage Error
res_wape = np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual))

# 6. Store and Display in Tabular Form
naive_results = [{
    "Subset": "Naive Seasonal Baseline",
    "MAE (kg)": round(res_mae, 3),
    "RMSE (kg)": round(res_rmse, 3),
    "WAPE": f"{res_wape * 100:.2f}%"
}]

naive_results_df = pd.DataFrame(naive_results)

print("\n--- Baseline Model Comparison ---")
display(naive_results_df)

## 4.2. Statistical Model: ARIMAX

As discovered in the literature review, ARIMA and its variations still show respectable results for different forecasting problems. Therefore we will se how it performs for our problem.

In [ ]:
# Train & evaluate the ARIMA models on all feature subsets

warnings.filterwarnings('ignore')

def calculate_wape(actual_ts, forecast_ts):
    """
    Manually calculates WAPE to handle zero-sales days safely.
    Formula: Sum of Absolute Errors / Sum of Actuals
    """
    actuals = actual_ts.values().flatten()
    forecasts = forecast_ts.values().flatten()
    
    total_abs_error = np.sum(np.abs(actuals - forecasts))
    total_volume = np.sum(actuals)
    
    # Avoid division by zero if the entire validation period has 0 sales
    if total_volume == 0:
        return np.nan 
        
    return total_abs_error / total_volume



# 1. Glue Train and Val covariates together so 'predict' has the info it needs
# Darts is smart: it will use the Train part for 'fit' and the Val part for 'predict'
X_s_joined = concatenate([X_train_s_ts, X_val_s_ts])
X_s_w_joined = concatenate([X_train_s_w_ts, X_val_s_w_ts])
X_s_c_joined = concatenate([X_train_s_c_ts, X_val_s_c_ts])
X_s_w_c_joined = concatenate([X_train_s_w_c_ts, X_val_s_w_c_ts])

# 2. Update the loop to use the joined series
subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]

results = []
print("🚀 Starting ARIMAX evaluation loop (with joined covariates)...")

for name, x_joined in subsets:
    try:
        model = ARIMA(p=1, d=1, q=1)
        
        # Fit using only the training portion of the target
        # Darts handles the alignment of x_joined automatically
        model.fit(y_train_ts, future_covariates=x_joined)
        
        # Predict n steps (length of validation set)
        prediction = model.predict(n=len(y_val_ts), future_covariates=x_joined)
        
        # Calculate Metrics
        res_mae = mae(y_val_ts, prediction)
        res_rmse = rmse(y_val_ts, prediction)
        res_wape = calculate_wape(y_val_ts, prediction)
        
        results.append({
            "Subset": name,
            "MAE (kg)": round(res_mae, 3),
            "RMSE (kg)": round(res_rmse, 3),
            "WAPE": f"{res_wape * 100:.2f}%"
        })
        print(f"✅ Finished {name}")
    except Exception as e:
        print(f"❌ Error in {name}: {e}")

# 3. Display results - This will now work as 'results' won't be empty!
results_df = pd.DataFrame(results)
if not results_df.empty:
    print("\n--- Model Comparison Table ---")
    display(results_df.sort_values(by="WAPE"))

In [ ]:
# Create plots of actual value vs. predicted value in order to understand better the deviations

# 1. Define the subsets and their joined covariates
subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]

# 2. Setup the figure
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)
fig.subplots_adjust(hspace=0.3)

print("📊 Generating plots for each subset...")

for i, (name, x_joined) in enumerate(subsets):
    # Initialize and Fit
    model = ARIMA(p=1, d=1, q=1)
    model.fit(y_train_ts, future_covariates=x_joined)
    
    # Predict for the validation period
    prediction = model.predict(n=len(y_val_ts), future_covariates=x_joined)
    
    # Plotting on the specific subplot axis
    # Convert to pandas for easier plotting control if desired, 
    # but Darts .plot() works directly with axes
    y_val_ts.plot(ax=axes[i], label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    prediction.plot(ax=axes[i], label=f'Predicted ({name})', color='crimson', lw=2, linestyle='--')
    
    axes[i].set_title(f"ARIMAX Forecast vs Actual: {name}", fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Weight (kg)")
    axes[i].legend(loc='upper left')
    axes[i].grid(True, alpha=0.3)

plt.xlabel("Date")
plt.savefig("arimax_comparison_plots.png", bbox_inches='tight', dpi=150)
print("✅ Visualization saved as 'arimax_comparison_plots.png'")

## 4.3. Tree-Based Models: LightGBM & XGBoost

The literature reveals that tree-based models show very good results for various time-series forecasting tasks. Therefore we will train LightGBM and XGBoost, two of the most popular and best performing tree-based algorithms.

In [ ]:
# Train and evaluate the LightGBM models on all feature subsets

subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]
lgbm_results = []

print("🚀 Starting LightGBM evaluation loop (Cleaned Version)...")

for name, x_joined in subsets:
    try:
        # Use a simpler initialization to avoid internal 'ambiguous' checks
        model_lgbm = LightGBMModel(
            lags=14, 
            lags_future_covariates=[0], 
            output_chunk_length=1,
            random_state=42,
            # Adding these to ensure the booster behaves
            importance_type='gain', 
            verbose=-1
        )
        
        # Ensure we are passing the series directly
        model_lgbm.fit(y_train_ts, future_covariates=x_joined)
        
        # Prediction
        prediction = model_lgbm.predict(n=len(y_val_ts), future_covariates=x_joined)
        
        # Enforce non-negativity
        prediction = prediction.map(lambda x: np.maximum(0, x))
        
        # Calculate Metrics
        res_mae = mae(y_val_ts, prediction)
        res_rmse = rmse(y_val_ts, prediction)
        res_wape = calculate_wape(y_val_ts, prediction)
        
        lgbm_results.append({
            "Subset": name,
            "MAE (kg)": round(res_mae, 3),
            "RMSE (kg)": round(res_rmse, 3),
            "WAPE": f"{res_wape * 100:.2f}%",
            "model_obj": model_lgbm # Save the model for feature importance later
        })
        print(f"✅ Finished {name}")
        
    except Exception as e:
        print(f"❌ Error in {name}: {str(e)}")

# Safe Display
if lgbm_results:
    lgbm_results_df = pd.DataFrame(lgbm_results).drop(columns=['model_obj'])
    print("\n--- LightGBM Model Comparison ---")
    display(lgbm_results_df.sort_values(by="WAPE"))

In [ ]:
# Extract the V3 model specifically from the results list

best_model = None
for result in lgbm_results:
    if result["Subset"] == "Sales + Calendric (V3)":
        best_model = result["model_obj"]
        print("✅ Found the V3 champion model!")
        break

if best_model is None:
    print("❌ Could not find V3 in results. Did the loop finish successfully?")

In [ ]:
# Let's check feature importance, to get a feeling for the most influential features

# 1. Get the feature names from the Darts model wrapper
# These names will look like 'EX_WEIGHT_lag-1' or 'temp_max_0'
darts_feature_names = best_model.lagged_feature_names

# 2. Extract importance from the underlying booster
booster = best_model.model
importances = booster.feature_importances_

# 3. Create a clean DataFrame
importance_df = pd.DataFrame({
    'Feature_Lag': darts_feature_names,
    'Importance_Gain': importances
}).sort_values(by='Importance_Gain', ascending=False)

# 4. Plotting
plt.figure(figsize=(12, 8))
top_n = 20
plt.barh(importance_df['Feature_Lag'].head(top_n), 
         importance_df['Importance_Gain'].head(top_n), 
         color='skyblue')
plt.gca().invert_yaxis()
plt.title(f'Top {top_n} Factors Driving Strawberry Sales (Decoded)')
plt.xlabel('Total Contribution to Model Accuracy (Gain)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Visualizing the predictions of LightGBM

# 1. Define the subsets
subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]

# 2. Setup the figure
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)
fig.subplots_adjust(hspace=0.3)

print("📊 Generating LightGBM plots for each subset...")

for i, (name, x_joined) in enumerate(subsets):
    try:
        # Initialize LightGBM with the same parameters used in the loop
        model_lgbm = LightGBMModel(
            lags=14, 
            lags_future_covariates=[0], 
            output_chunk_length=1,
            random_state=42,
            verbose=-1
        )
        
        # Fit on training data
        model_lgbm.fit(y_train_ts, future_covariates=x_joined)
        
        # Predict for the validation period
        prediction = model_lgbm.predict(n=len(y_val_ts), future_covariates=x_joined)
        
        # Enforce the "No Negative Sales" rule for the plot
        prediction = prediction.map(lambda x: np.maximum(0, x))
        
        # Plotting
        y_val_ts.plot(ax=axes[i], label='Actual (EX_WEIGHT)', color='black', lw=1.5)
        prediction.plot(ax=axes[i], label=f'Predicted ({name})', color='forestgreen', lw=2, linestyle='--')
        
        axes[i].set_title(f"LightGBM Forecast vs Actual: {name}", fontsize=14, fontweight='bold')
        axes[i].set_ylabel("Weight (kg)")
        axes[i].legend(loc='upper left')
        axes[i].grid(True, alpha=0.3)
        print(f"✅ Plotted {name}")
        
    except Exception as e:
        axes[i].set_title(f"Error in {name}")
        print(f"❌ Error plotting {name}: {e}")

plt.xlabel("Date")
plt.savefig("lgbm_comparison_plots.png", bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualization saved as 'lgbm_comparison_plots.png'")

In [ ]:
# Train and evaluate XGBoost models on all feature subsets

subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]
xgb_results = []

print("🚀 Starting XGBoost evaluation loop (Optimized for Intermittent Demand)...")

for name, x_joined in subsets:
    try:
        # XGBModel implementation in Darts
        model_xgb = XGBModel(
            lags=14, 
            lags_future_covariates=[0], 
            output_chunk_length=1,
            random_state=42,
            # 'objective': 'reg:tweedie' is excellent for data with many zeros
            # 'tweedie_variance_power': 1.5 is a standard starting point for demand
            objective='reg:tweedie',
            tweedie_variance_power=1.5,
            n_jobs=-1 # Use all available CPU cores
        )
        
        # Train the model
        model_xgb.fit(y_train_ts, future_covariates=x_joined)
        
        # Prediction
        prediction = model_xgb.predict(n=len(y_val_ts), future_covariates=x_joined)
        
        # Enforce non-negativity
        prediction = prediction.map(lambda x: np.maximum(0, x))
        
        # Calculate Metrics (using your existing mae, rmse, and calculate_wape functions)
        res_mae = mae(y_val_ts, prediction)
        res_rmse = rmse(y_val_ts, prediction)
        res_wape = calculate_wape(y_val_ts, prediction)
        
        xgb_results.append({
            "Subset": name,
            "MAE (kg)": round(res_mae, 3),
            "RMSE (kg)": round(res_rmse, 3),
            "WAPE": f"{res_wape * 100:.2f}%",
            "model_obj": model_xgb
        })
        print(f"✅ Finished XGBoost: {name}")
        
    except Exception as e:
        print(f"❌ Error in {name}: {str(e)}")

# Display Results
if xgb_results:
    xgb_results_df = pd.DataFrame(xgb_results).drop(columns=['model_obj'])
    print("\n--- XGBoost Model Comparison ---")
    display(xgb_results_df.sort_values(by="WAPE"))

In [ ]:
# Visualize the results

# 1. Define the subsets (Assuming X_s_joined, etc., are already defined)
subsets = [
    ("Sales Only (V1)", X_s_joined),
    ("Sales + Weather (V2)", X_s_w_joined),
    ("Sales + Calendric (V3)", X_s_c_joined),
    ("Full Set (V4)", X_s_w_c_joined)
]

# 2. Setup the figure
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)
fig.subplots_adjust(hspace=0.3)

print("📊 Generating XGBoost plots for each subset...")

for i, (name, x_joined) in enumerate(subsets):
    try:
        # Initialize XGBoost with the optimized Tweedie parameters
        model_xgb = XGBModel(
            lags=14, 
            lags_future_covariates=[0], 
            output_chunk_length=1,
            random_state=42,
            objective='reg:tweedie',      # Better for data with many zeros
            tweedie_variance_power=1.5,
            n_jobs=-1,
            verbosity=0
        )
        
        # Fit on training data
        model_xgb.fit(y_train_ts, future_covariates=x_joined)
        
        # Predict for the validation period
        prediction = model_xgb.predict(n=len(y_val_ts), future_covariates=x_joined)
        
        # --- Apply Post-Processing ---
        # A. Enforce Non-Negativity
        prediction = prediction.map(lambda x: np.maximum(0, x))
        
        # B. Enforce Seasonal Zeroing (Business Logic)
        pred_values = prediction.values().flatten()
        months = prediction.time_index.month
        for j, month in enumerate(months):
            if month in off_season:
                pred_values[j] = 0.0
        
        # Re-wrap values back into the TimeSeries
        final_prediction = prediction.with_values(pred_values.reshape(-1, 1, 1))
        
        # --- Plotting ---
        y_val_ts.plot(ax=axes[i], label='Actual (EX_WEIGHT)', color='black', lw=1.5)
        final_prediction.plot(ax=axes[i], label=f'XGBoost Predicted ({name})', 
                             color='darkorange', lw=2, linestyle='--')
        
        axes[i].set_title(f"XGBoost Forecast vs Actual: {name}", fontsize=14, fontweight='bold')
        axes[i].set_ylabel("Weight (kg)")
        axes[i].legend(loc='upper left')
        axes[i].grid(True, alpha=0.3)
        print(f"✅ Plotted {name}")
        
    except Exception as e:
        axes[i].set_title(f"Error in {name}")
        print(f"❌ Error plotting {name}: {e}")

plt.xlabel("Date")
plt.savefig("xgb_comparison_plots.png", bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualization saved as 'xgb_comparison_plots.png'")

## 4.4. Neural Network: LSTM

Also neural networks prove in several researches, that they can be a good choice for time-series forecasting tasks, especially LSTM. Therefore, we also create models based on LSTM and evaluate their performance.

In [ ]:
# As NNs work differently than tree-based models, we need to remove the manually created lag features from the dataframes

# List of manual lag/rolling features to remove
features_to_remove = ['lag_7', 'lag_14', 'rolling_mean_7']

def clean_covariates(ts_object, remove_list):
    """
    Returns a TimeSeries with only the original features, 
    removing the manually created lags.
    """
    # Find all current columns
    all_cols = ts_object.components
    # Filter out the ones in our removal list
    keep_cols = [c for c in all_cols if c not in remove_list]
    # Return the sliced TimeSeries
    return ts_object[keep_cols]

# Apply cleaning to your 4 joined subsets
X_s_clean = clean_covariates(X_s_joined, features_to_remove)
X_s_w_clean = clean_covariates(X_s_w_joined, features_to_remove)
X_s_c_clean = clean_covariates(X_s_c_joined, features_to_remove)
X_s_w_c_clean = clean_covariates(X_s_w_c_joined, features_to_remove)

# Verify the result
print(f"Original columns in Full Set: {len(X_s_w_c_joined.components)}")
print(f"Cleaned columns for LSTM: {len(X_s_w_c_clean.components)}")
print(f"Features remaining: {X_s_w_c_clean.components.tolist()}")

In [ ]:
# Train the LSTM models with all feature subsets and get their results on the validation set

# 1. Define the cleaned subsets (Lags removed)
subsets_clean = [
    ("Sales Only (V1)", X_s_clean),
    ("Sales + Weather (V2)", X_s_w_clean),
    ("Sales + Calendric (V3)", X_s_c_clean),
    ("Full Set (V4)", X_s_w_c_clean)
]

lstm_results = []

print("🧠 Starting LSTM evaluation loop with Inverse Scaling...")

for name, x_clean_joined in subsets_clean:
    try:
        model_lstm = RNNModel(
            model="LSTM",
            hidden_dim=30,
            n_rnn_layers=2,
            dropout=0.15,
            batch_size=16,
            n_epochs=100,
            optimizer_kwargs={"lr": 1e-3},
            input_chunk_length=14,
            training_length=21,
            random_state=42,
            force_reset=True,
            pl_trainer_kwargs={"accelerator": "cpu"}
        )
        
        # Train using the SCALED target
        model_lstm.fit(
            series=y_train_scaled_ts, 
            future_covariates=x_clean_joined,
            verbose=False
        )
        
        # 1. Predict (Result will be scaled)
        prediction_scaled = model_lstm.predict(n=len(y_val_scaled_ts), future_covariates=x_clean_joined)
        
        # 2. Invert Scaling to get back to Kilograms
        # This makes metrics like MAE and RMSE human-readable again
        # Convert Darts TimeSeries values to NumPy, reshape to 2D for sklearn, then convert back
        raw_values = prediction_scaled.values().reshape(-1, 1)
        unscaled_values = y_scaler.inverse_transform(raw_values)

        # Re-create the TimeSeries object so you can plot it easily
        prediction_kg = prediction_scaled.with_values(unscaled_values)
        # 3. Enforce non-negativity (Physical reality check)
        prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
        
        # 4. Calculate Metrics against the ORIGINAL (unscaled) validation set
        # This ensures the comparison with LightGBM is 1:1
        res_mae = mae(y_val_ts, prediction_kg)
        res_rmse = rmse(y_val_ts, prediction_kg)
        res_wape = calculate_wape(y_val_ts, prediction_kg)
        
        lstm_results.append({
            "Subset": name,
            "MAE (kg)": round(res_mae, 3),
            "RMSE (kg)": round(res_rmse, 3),
            "WAPE": f"{res_wape * 100:.2f}%",
            "model_obj": model_lstm
        })
        print(f"✅ Finished LSTM: {name}")
        
    except Exception as e:
        print(f"❌ Error in {name}: {e}")

# 2. Display results
lstm_results_df = pd.DataFrame(lstm_results)
if not lstm_results_df.empty:
    print("\n--- LSTM Model Comparison (Values in Kilograms) ---")
    display(lstm_results_df.sort_values(by="WAPE"))

In [ ]:
# Visualize the results

# 1. Setup the figure
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)
fig.subplots_adjust(hspace=0.3)

print("📊 Generating LSTM plots using stored models...")

# We iterate through the results list you already created during the 16-min run
# This assumes subsets_clean and lstm_results are in your current memory
for i, result in enumerate(lstm_results):
    name = result["Subset"]
    
    # Get the corresponding covariate set from our earlier clean list
    # Match the name to find the right X series
    x_clean_joined = next(x for n, x in subsets_clean if n == name)
    
    try:
        # 1. Use the model saved during the evaluation loop
        # Ensure you added "model_obj": model_lstm to your lstm_results.append()
        current_model = result["model_obj"]
        
        # 2. Predict (This is fast, retraining is what took 16 mins)
        prediction_scaled = current_model.predict(n=len(y_val_scaled_ts), 
                                                future_covariates=x_clean_joined)
        
        # Mirror the fix you used in the training loop:
        raw_vals = prediction_scaled.values().reshape(-1, 1)
        unscaled_vals = y_scaler.inverse_transform(raw_vals)
        prediction_kg = prediction_scaled.with_values(unscaled_vals)
    
        # Enforce non-negativity
        prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
        
        # 5. Plotting on the specific subplot axis
        # Plot the ORIGINAL unscaled validation set for comparison
        y_val_ts.plot(ax=axes[i], label='Actual (EX_WEIGHT)', color='black', lw=1.5)
        prediction_kg.plot(ax=axes[i], label=f'LSTM Predicted ({name})', color='darkviolet', lw=2, linestyle='--')
        
        axes[i].set_title(f"LSTM Forecast vs Actual: {name} (WAPE: {result['WAPE']})", fontsize=14, fontweight='bold')
        axes[i].set_ylabel("Weight (kg)")
        axes[i].legend(loc='upper left')
        axes[i].grid(True, alpha=0.3)
        print(f"✅ Plotted {name}")
        
    except Exception as e:
        axes[i].set_title(f"Error in {name}")
        print(f"❌ Could not plot {name}: {e}. (Did you save 'model_obj' in lstm_results?)")

plt.xlabel("Date")
plt.savefig("lstm_comparison_plots.png", bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualization saved as 'lstm_comparison_plots.png'")

## 4.5. Transformer: TFT

Lastly, we will deal with the most recent and most complex architecture in scope of this research: Transformers. Specifically the TFT proved to reach high accuracy in different research experiments, so we want to see how it perfroms for our problem.

In [ ]:
# Train the TFT models with all feature subsets and get their results on the validation set

# 1. Early Stopping is your best friend on CPU to save time!
my_stopper = EarlyStopping(
    monitor="train_loss",
    patience=10,        # Stop if no improvement for 10 epochs
    min_delta=0.0005,
    mode="min",
)

STOCKOUT_COL = 'is_stockout'
subsets_clean = [
    ("Sales Only (V1)", X_s_clean),
    ("Sales + Weather (V2)", X_s_w_clean),
    ("Sales + Calendric (V3)", X_s_c_clean),
    ("Full Set (V4)", X_s_w_c_clean)
]

tft_results = []

print("⚙️ GPU not detected. Switching to CPU mode with Early Stopping...")

for name, x_combined in subsets_clean:
    try:
        # --- Separate Past (Stockout) and Future (Everything else) ---
        if STOCKOUT_COL in x_combined.components:
            past_covs = x_combined[STOCKOUT_COL]
            future_covs = x_combined[[c for c in x_combined.components if c != STOCKOUT_COL]]
        else:
            past_covs = None
            future_covs = x_combined

        # 2. Define the TFT Model (Back to CPU)
        model_tft = TFTModel(
            input_chunk_length=14,
            output_chunk_length=7,
            hidden_size=16,
            lstm_layers=2,
            num_attention_heads=4,
            dropout=0.1,
            batch_size=16,
            n_epochs=100, 
            add_relative_index=True,
            random_state=44,
            force_reset=True,
            pl_trainer_kwargs={
                "accelerator": "cpu", # <--- Fixed for your environment
                "callbacks": [my_stopper]
            }
        )
        
        # 3. Train
        model_tft.fit(
            series=y_train_scaled_ts, 
            future_covariates=future_covs,
            past_covariates=past_covs,
            verbose=False
        )
        
        # 4. Predict
        prediction_scaled = model_tft.predict(
            n=len(y_val_scaled_ts), 
            series=y_train_scaled_ts,
            future_covariates=future_covs,
            past_covariates=past_covs
        )
        
        # 5. Manual Unscaling Fix
        raw_values = prediction_scaled.values().reshape(-1, 1)
        unscaled_values = y_scaler.inverse_transform(raw_values)
        prediction_kg = prediction_scaled.with_values(unscaled_values)
        prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
        
        # 6. Metrics
        res_mae = mae(y_val_ts, prediction_kg)
        res_rmse = rmse(y_val_ts, prediction_kg)
        res_wape = calculate_wape(y_val_ts, prediction_kg)
        
        tft_results.append({
            "Subset": name,
            "MAE (kg)": round(res_mae, 3),
            "RMSE (kg)": round(res_rmse, 3),
            "WAPE": f"{res_wape * 100:.2f}%",
            "model_obj": model_tft
        })
        print(f"✅ Finished TFT: {name}")
        
    except Exception as e:
        print(f"❌ Error in TFT {name}: {e}")

# --- Display Results ---
tft_results_df = pd.DataFrame(tft_results)
if not tft_results_df.empty:
    print("\n--- TFT Model Comparison (CPU Mode) ---")
    display(tft_results_df.sort_values(by="WAPE"))

In [ ]:
# Visualize the results

# 1. Setup the figure (4 subplots for the 4 versions)
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)
fig.subplots_adjust(hspace=0.3)

print("📊 Generating TFT plots from stored models...")

# Iterate through the results we just generated
for i, result in enumerate(tft_results):
    name = result["Subset"]
    
    # Match the name to find the right covariate series from our earlier clean list
    x_combined = next(x for n, x in subsets_clean if n == name)
    
    try:
        # 1. Retrieve the trained TFT model
        current_model = result["model_obj"]
        
        # 2. Handle Past/Future split exactly as we did in training
        if STOCKOUT_COL in x_combined.components:
            past_covs = x_combined[STOCKOUT_COL]
            future_covs = x_combined[[c for c in x_combined.components if c != STOCKOUT_COL]]
        else:
            past_covs = None
            future_covs = x_combined
            
        # 3. Predict (This is fast since the model is already trained)
        prediction_scaled = current_model.predict(
            n=len(y_val_scaled_ts), 
            series=y_train_scaled_ts,
            future_covariates=future_covs,
            past_covariates=past_covs
        )
        
        # 4. Manual Unscaling Fix (Same as training loop)
        raw_vals = prediction_scaled.values().reshape(-1, 1)
        unscaled_vals = y_scaler.inverse_transform(raw_vals)
        prediction_kg = prediction_scaled.with_values(unscaled_vals)
        
        # 5. Enforce non-negativity
        prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
        
        # 6. Plotting
        # Plot Actuals (using your unscaled validation series)
        y_val_ts.plot(ax=axes[i], label='Actual (EX_WEIGHT)', color='black', lw=1.5)
        
        # Plot TFT Prediction
        prediction_kg.plot(ax=axes[i], label=f'TFT Predicted ({name})', color='royalblue', lw=2, linestyle='--')
        
        # Formatting
        axes[i].set_title(f"TFT Forecast vs Actual: {name} (WAPE: {result['WAPE']})", fontsize=14, fontweight='bold')
        axes[i].set_ylabel("Weight (kg)")
        axes[i].legend(loc='upper left')
        axes[i].grid(True, alpha=0.3)
        
        print(f"✅ Plotted {name}")
        
    except Exception as e:
        axes[i].set_title(f"Error in {name}")
        print(f"❌ Could not plot {name}: {e}")

plt.xlabel("Date")
plt.savefig("tft_comparison_plots.png", bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualization saved as 'tft_comparison_plots.png'")

# 5. Modeling: Hyperparameter Tuning

For each algorithm that was trained above, the model with the best performing feature-subset will be hyperparameter-tuned in order to improve performance to the maximum possible

## 5.1. ARIMAX: Full Set V4

In [ ]:
# Perform hyperparameter tuning and evaluate results on validation set

warnings.filterwarnings('ignore')

# --- 1. Custom WAPE Metric ---
def calculate_wape(actual_ts, forecast_ts):
    """
    Manually calculates WAPE to handle zero-sales days safely.
    """
    actuals = actual_ts.values().flatten()
    forecasts = forecast_ts.values().flatten()
    
    total_abs_error = np.sum(np.abs(actuals - forecasts))
    total_volume = np.sum(actuals)
    
    if total_volume == 0:
        return np.nan 
        
    return total_abs_error / total_volume


# --- Updated Custom Metric with Physical Constraints ---
def calculate_wape_with_constraints(actual_ts, forecast_ts):
    # 1. Apply Non-Negativity
    forecast_constrained = forecast_ts.map(lambda x: np.maximum(0, x))
    
    # 2. Apply Seasonal Zeroing (Optional: Only if business logic is 100% sure)
    # This zeroes out months 1, 2, 3, 11, and 12
    off_season = [1, 2, 3, 11, 12]
    
    # We convert to values for fast manipulation
    forecast_values = forecast_constrained.values().flatten()
    months = forecast_constrained.time_index.month
    
    for i, month in enumerate(months):
        if month in off_season:
            forecast_values[i] = 0.0
            
    # Re-wrap for metric calculation
    final_forecast = forecast_constrained.with_values(forecast_values.reshape(-1, 1, 1))
    
    # Calculate WAPE
    actuals = actual_ts.values().flatten()
    forecasts = final_forecast.values().flatten()
    
    total_abs_error = np.sum(np.abs(actuals - forecasts))
    total_volume = np.sum(actuals)
    
    return total_abs_error / total_volume if total_volume != 0 else np.nan

# --- 2. Grid Search Configuration ---
# Define the search space for p (AutoRegressive), d (Differencing), q (Moving Average)
# We focus on a search space that avoids over-complexity
parameters = {
    "p": [0, 1, 2, 3],
    "d": [0, 1],
    "q": [0, 1, 2, 3],
    "seasonal_order": [(0, 0, 0, 0)] # Seasonal order can be added if needed, e.g., (1, 1, 1, 7)
}

print("🔍 Starting ARIMAX Grid Search on Full Set (V4)...")

# Perform the Grid Search using WAPE as the optimization metric
# Gridsearch handles the training on y_train_ts and evaluating on y_val_ts automatically
best_model_arima, best_params, _ = ARIMA.gridsearch(
    parameters=parameters,
    series=y_train_ts,
    future_covariates=X_s_w_c_joined,
    val_series=y_val_ts,
    metric=calculate_wape_with_constraints, # Optimization objective
    n_jobs=-1,             # Use all available CPU cores
    verbose=False
)

print(f"✅ Grid Search Complete. Best Parameters: {best_params}")

# --- 3. Final Evaluation of the Best Model ---
# RE-FIT STEP: This fixes the ValueError by explicitly attaching the covariates
best_model_arima.fit(y_train_ts, future_covariates=X_s_w_c_joined)
# Since gridsearch returns the best model fitted on 'series', we predict the validation period
prediction = best_model_arima.predict(n=len(y_val_ts), future_covariates=X_s_w_c_joined)

# Calculate final metrics for the best solution
res_mae = mae(y_val_ts, prediction)
res_rmse = rmse(y_val_ts, prediction)
res_wape = calculate_wape_with_constraints(y_val_ts, prediction)

# --- 4. Display Results in Tabular Format ---
best_results = [{
    "Model": "ARIMAX (Tuned)",
    "Subset": "Full Set (V4)",
    "Best Parameters": str(best_params),
    "MAE (kg)": round(res_mae, 3),
    "RMSE (kg)": round(res_rmse, 3),
    "WAPE": f"{res_wape * 100:.2f}%"
}]

best_results_df = pd.DataFrame(best_results)

print("\n--- Best ARIMAX Solution (Tuned via WAPE) ---")
display(best_results_df)

In [ ]:
# Visualization of hyperparameter tuned ARIMAX model

# 1. Setup the figure for a single, clear plot
plt.figure(figsize=(15, 6))

# Define the off-season months for zeroing
off_season = [1, 2, 3, 11, 12]

try:
    # 2. Initialize and Re-fit the best model
    # We use **best_params to unpack p, d, q directly
    model_tuned = ARIMA(**best_params)
    model_tuned.fit(y_train_ts, future_covariates=X_s_w_c_joined)
    
    # 3. Predict for the validation period
    prediction = model_tuned.predict(n=len(y_val_ts), future_covariates=X_s_w_c_joined)
    
    # --- 4. Apply Post-Processing Constraints ---
    # Constraint A: Non-Negativity
    prediction = prediction.map(lambda x: np.maximum(0, x))
    
    # Constraint B: Seasonal Zeroing
    pred_values = prediction.values().flatten()
    months = prediction.time_index.month
    for j, month in enumerate(months):
        if month in off_season:
            pred_values[j] = 0.0
    
    # Update the series with constrained values
    final_prediction = prediction.with_values(pred_values.reshape(-1, 1, 1))
    
    # --- 5. Plotting ---
    y_val_ts.plot(label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    final_prediction.plot(label='Tuned ARIMAX (V4) + Constraints', color='crimson', lw=2, linestyle='--')
    
    plt.title("Final Tuned ARIMAX Performance: Full Set (V4)", fontsize=16, fontweight='bold')
    plt.ylabel("Weight (kg)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    # Save the specific plot
    plt.savefig("tuned_arimax_v4_final.png", bbox_inches='tight', dpi=150)
    plt.show()
    
    print(f"✅ Visualization of Tuned V4 complete. Best Params used: {best_params}")

except Exception as e:
    print(f"❌ Error during visualization: {e}")

## 5.2. LightGBM: Sales + Calendric Data V3

In [ ]:
# Perform hyperparameter tuning and evaluate results on validation set

# We use a broad range for Random Search to explore the space efficiently
param_grid = {
    "lags": [7, 14, 21, 28],
    "num_leaves": [20, 30, 40],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [50, 100, 200],
    # MUST BE FIXED VALUES (not lists) if you want all models to use them
    "lags_future_covariates": [[0]], 
    "output_chunk_length": [1],
    "random_state": [42]
}

print("🔍 Starting LightGBM Random Search on Subset V3...")

# Perform Random Search
# n_random_samples: Determines how many combinations to try (higher = better but slower)
# For 'minutes, not hours', 20-30 samples is usually the sweet spot.
best_model_lgbm, best_params_lgbm, _ = LightGBMModel.gridsearch(
    parameters=param_grid,
    series=y_train_ts,
    future_covariates=X_s_c_joined, # Using V3 (Calendric)
    val_series=y_val_ts,
    metric=calculate_wape_with_constraints,
    n_random_samples=100, 
    n_jobs=-1,
    random_state=42,
    verbose=False
)

print(f"✅ Best Parameters: {best_params_lgbm}")

# --- Final Evaluation ---
# RE-FIT: This step is mandatory to provide the 'series' the error is asking for
best_model_lgbm.fit(y_train_ts, future_covariates=X_s_c_joined)

# Generate prediction with best model
prediction = best_model_lgbm.predict(n=len(y_val_ts), future_covariates=X_s_c_joined)

# Apply final constraints for reporting
pred_values = prediction.values().flatten()
months = prediction.time_index.month
for i, m in enumerate(months):
    if m in [1, 2, 3, 11, 12]: pred_values[i] = 0.0
final_pred = prediction.with_values(pred_values.reshape(-1, 1, 1))

# Metrics
res_mae = mae(y_val_ts, final_pred)
res_rmse = rmse(y_val_ts, final_pred)
res_wape = calculate_wape_with_constraints(y_val_ts, prediction)

# --- Display Result ---
lgbm_tuned_results = [{
    "Model": "LightGBM (Tuned)",
    "Subset": "Sales + Calendric (V3)",
    "MAE (kg)": round(res_mae, 3),
    "RMSE (kg)": round(res_rmse, 3),
    "WAPE": f"{res_wape * 100:.2f}%"
}]

display(pd.DataFrame(lgbm_tuned_results))

In [ ]:
# Visualize the results

# 1. Setup the figure
plt.figure(figsize=(15, 6))

# Define the off-season months (Jan, Feb, Mar, Nov, Dec)
off_season = [1, 2, 3, 11, 12]

try:
    # 2. Initialize and Re-fit the best model
    # We unpack best_params (lags, num_leaves, etc.) directly
    model_lgbm_tuned = LightGBMModel(**best_params_lgbm)
    
    # We must re-fit because the search result is often a 'clean' object
    model_lgbm_tuned.fit(y_train_ts, future_covariates=X_s_c_joined)
    
    # 3. Predict for the validation period (n matches length of y_val_ts)
    prediction = model_lgbm_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined)
    
    # --- 4. Apply Post-Processing Constraints ---
    # Constraint A: Non-Negativity (LightGBM usually does this, but let's be safe)
    prediction = prediction.map(lambda x: np.maximum(0, x))
    
    # Constraint B: Seasonal Zeroing for the Off-Season
    pred_values = prediction.values().flatten()
    months = prediction.time_index.month
    for j, month in enumerate(months):
        if month in off_season:
            pred_values[j] = 0.0
    
    # Update the series with constrained values
    final_prediction = prediction.with_values(pred_values.reshape(-1, 1, 1))
    
    # --- 5. Plotting ---
    y_val_ts.plot(label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    final_prediction.plot(label='Tuned LightGBM (V3) + Constraints', color='forestgreen', lw=2, linestyle='--')
    
    plt.title(f"Final Tuned LightGBM Performance: Sales + Calendric (V3)", fontsize=16, fontweight='bold')
    plt.ylabel("Weight (kg)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    # Save the specific plot
    plt.savefig("tuned_lgbm_v3_final.png", bbox_inches='tight', dpi=150)
    plt.show()
    
    print(f"✅ Visualization of Tuned V3 complete.")
    print(f"📌 Best Params used: {best_params_lgbm}")

except Exception as e:
    print(f"❌ Error during visualization: {e}")

## 5.3. XGBoost: Sales + Calendric Data V3

In [ ]:
# Perform hyperparameter tuning and evaluate results on validation set

# --- 1. Random Search Parameter Grid ---
# We include 'tweedie_variance_power' to find the best zero-handling balance
xgb_param_grid = {
    "lags": [7, 14, 21, 28],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200],
    "tweedie_variance_power": [1.1, 1.5, 1.9], # Tuning the distribution shape
    "lags_future_covariates": [[0]],
    "output_chunk_length": [1],
    "random_state": [42],
    "objective": ["reg:tweedie"]
}

print("🔍 Starting XGBoost Random Search (Tweedie Optimized) on V3...")

# Perform Random Search
best_model_xgb, best_params_xgb, _ = XGBModel.gridsearch(
    parameters=xgb_param_grid,
    series=y_train_ts,
    future_covariates=X_s_c_joined, # Using V3
    val_series=y_val_ts,
    metric=calculate_wape_with_constraints,        # Using your custom metric from LightGBM
    n_random_samples=100, 
    n_jobs=-1,
    random_state=42,
    verbose=False
)

# --- 2. Re-fit and Final Evaluation ---
print(f"✅ Best XGBoost Params: {best_params_xgb}")

# Mandatory re-fit to avoid the 'series' error
best_model_xgb.fit(y_train_ts, future_covariates=X_s_c_joined)

# Prediction
prediction_xgb = best_model_xgb.predict(n=len(y_val_ts), future_covariates=X_s_c_joined)

# Apply Seasonal Zeroing (Jan, Feb, Mar, Nov, Dec)
off_season = [1, 2, 3, 11, 12]
pred_vals = prediction_xgb.values().flatten()
months = prediction_xgb.time_index.month
for i, m in enumerate(months):
    if m in off_season: pred_vals[i] = 0.0
final_pred_xgb = prediction_xgb.with_values(pred_vals.reshape(-1, 1, 1))

# Final Metrics
res_mae_xgb = mae(y_val_ts, final_pred_xgb)
res_rmse_xgb = rmse(y_val_ts, final_pred_xgb)
res_wape_xgb = calculate_wape_with_constraints(y_val_ts, final_pred_xgb)

# --- 2. Prepare the results in the standardized format ---
xgb_tuned_results = [{
    "Model": "XGBoost (Tuned)",
    "Subset": "Sales + Calendric (V3)",
    "MAE (kg)": round(res_mae_xgb, 3),
    "RMSE (kg)": round(res_rmse_xgb, 3),
    "WAPE": f"{res_wape_xgb * 100:.2f}%"
}]

# --- 3. Display the standardized Table ---
display(pd.DataFrame(xgb_tuned_results))

In [ ]:
# Visualize the results

# 1. Setup the figure
plt.figure(figsize=(15, 6))

# Define the off-season months (Jan, Feb, Mar, Nov, Dec)
off_season = [1, 2, 3, 11, 12]

try:
    # 2. Initialize and Re-fit the best XGBoost model
    # Use best_params_xgb (lags, max_depth, tweedie_variance_power, etc.)
    model_xgb_tuned = XGBModel(**best_params_xgb)
    
    # Re-fit to the training data and V3 covariates
    model_xgb_tuned.fit(y_train_ts, future_covariates=X_s_c_joined)
    
    # 3. Predict for the validation period
    prediction = model_xgb_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined)
    
    # --- 4. Apply Post-Processing Constraints ---
    # Constraint A: Non-Negativity
    prediction = prediction.map(lambda x: np.maximum(0, x))
    
    # Constraint B: Seasonal Zeroing for the Off-Season
    pred_values = prediction.values().flatten()
    months = prediction.time_index.month
    for j, month in enumerate(months):
        if month in off_season:
            pred_values[j] = 0.0
    
    # Update the series with final values
    final_prediction = prediction.with_values(pred_values.reshape(-1, 1, 1))
    
    # --- 5. Plotting ---
    y_val_ts.plot(label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    final_prediction.plot(label='Tuned XGBoost (V3) + Constraints', color='darkorange', lw=2, linestyle='--')
    
    plt.title("Final Tuned XGBoost Performance: Sales + Calendric (V3)", fontsize=16, fontweight='bold')
    plt.ylabel("Weight (kg)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    # Save the specific plot
    plt.savefig("tuned_xgb_v3_final.png", bbox_inches='tight', dpi=150)
    plt.show()
    
    print(f"✅ Visualization of Tuned XGBoost V3 complete.")
    print(f"📌 Best Params used: {best_params_xgb}")

except Exception as e:
    print(f"❌ Error during visualization: {e}")

## 5.4. LSTM: Full Set V4

In [ ]:
# Perform hyperparameter tuning and evaluate results on validation set

# --- 1. Parameter Grid for LSTM ---
# Using V4 settings with a focus on architecture and learning speed
lstm_param_grid = {
    "hidden_dim": [20, 30, 50],
    "n_rnn_layers": [1, 2],
    "dropout": [0.1, 0.2],
    "batch_size": [16, 32],
    "n_epochs": [100], 
    "optimizer_kwargs": [{"lr": 1e-3}, {"lr": 5e-3}],
    "input_chunk_length": [14, 21],
    "training_length": [28], # Must be >= input_chunk_length
    "random_state": [42],
    "force_reset": [True],
    "pl_trainer_kwargs": [{"accelerator": "cpu"}],
    "model": ["LSTM"]
}

print("🧠 Starting LSTM Random Search on V4 (Full Set - Scaled)...")

# Perform Random Search
# n_random_samples=8 keeps the time manageable (approx 30-50 mins depending on hardware)
best_model_lstm, best_params_lstm, _ = RNNModel.gridsearch(
    parameters=lstm_param_grid,
    series=y_train_scaled_ts,        # Training on SCALED data
    future_covariates=X_s_w_c_clean, # Using V4
    val_series=y_val_scaled_ts,      # Validating on SCALED data
    metric=calculate_wape_with_constraints, 
    n_random_samples=20, 
    n_jobs=1,                        # RNNs prefer sequential execution on CPU
    verbose=False
)

# --- 2. Re-fit and Inverse Scaling Evaluation ---
print(f"✅ Best LSTM Params: {best_params_lstm}")

# Re-fit the best model
best_model_lstm.fit(series=y_train_scaled_ts, future_covariates=X_s_w_c_clean)

# Predict (Result is still scaled)
prediction_scaled = best_model_lstm.predict(n=len(y_val_scaled_ts), future_covariates=X_s_w_c_clean)

# --- 3. RESCALING to Kilograms ---
raw_values = prediction_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_kg = prediction_scaled.with_values(unscaled_values)

# Apply Seasonal Zeroing (Business Logic)
off_season = [1, 2, 3, 11, 12]
pred_vals_kg = prediction_kg.values().flatten()
months = prediction_kg.time_index.month
for i, m in enumerate(months):
    if m in off_season:
        pred_vals_kg[i] = 0.0
final_pred_lstm = prediction_kg.with_values(pred_vals_kg.reshape(-1, 1, 1))
final_pred_lstm = final_pred_lstm.map(lambda x: np.maximum(0, x))

# --- 4. Final Standardized Metrics ---
res_mae_lstm = mae(y_val_ts, final_pred_lstm)
res_rmse_lstm = rmse(y_val_ts, final_pred_lstm)
res_wape_lstm = calculate_wape_with_constraints(y_val_ts, final_pred_lstm)

lstm_tuned_results = [{
    "Model": "LSTM (Tuned)",
    "Subset": "Full Set (V4)",
    "MAE (kg)": round(res_mae_lstm, 3),
    "RMSE (kg)": round(res_rmse_lstm, 3),
    "WAPE": f"{res_wape_lstm * 100:.2f}%"
}]

display(pd.DataFrame(lstm_tuned_results))

In [ ]:
# Retrain best model to include random state for reproducibility

best_params_lstm["random_state"] = 42

# 2. Re-initialize the model using the best params from your gridsearch
# This ensures it's a fresh object with no 'memory' of previous fits
best_model_lstm = RNNModel(
    **best_params_lstm,
)

print("🗼 Re-fitting LSTM using the same logic as Gridsearch...")

# 3. Fit using the FULL covariate series (X_s_w_c_clean) 
# Darts will slice this to match the training/validation series automatically.
best_model_lstm.fit(
    series=y_train_scaled_ts, 
    future_covariates=X_s_w_c_clean, # Use the full series, not a subset
    val_series=y_val_scaled_ts,
    val_future_covariates=X_s_w_c_clean # Pass the same full series here
)

print("✅ Fit successful! Dimensions are now aligned.")

# --- 3. Validation Prediction (Scaled) ---
# Use the end of the training set as context to predict the validation period
prediction_scaled = best_model_lstm.predict(
    n=len(y_val_scaled_ts), 
    series=y_train_scaled_ts, 
    future_covariates=X_s_w_c_clean
)

# --- 4. Post-Processing: Unscaling and Constraints ---
# Inverse Scaling
raw_values = prediction_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_kg = prediction_scaled.with_values(unscaled_values)

# Apply Physical Constraints (Function already defined in your environment)
final_pred_lstm_val = apply_strawberry_constraints(prediction_kg)

# --- 5. Updated Standardized Metrics ---
res_mae_lstm = mae(y_val_ts, final_pred_lstm_val)
res_rmse_lstm = rmse(y_val_ts, final_pred_lstm_val)
res_wape_lstm = calculate_wape_with_constraints(y_val_ts, final_pred_lstm_val)

lstm_tuned_results = [{
    "Model": "LSTM (Tuned - Deterministic)",
    "Subset": "Full Set (V4)",
    "MAE (kg)": round(res_mae_lstm, 3),
    "RMSE (kg)": round(res_rmse_lstm, 3),
    "WAPE": f"{res_wape_lstm * 100:.2f}%"
}]

print("✅ LSTM Re-training Complete.")
display(pd.DataFrame(lstm_tuned_results))

In [ ]:
# Save the model so the long training of the model doesn't have to be repeated

best_model_lstm.save("best_lstm_v4_trained_20260301.pt")

In [ ]:
# Visualize the results

# 1. Setup the figure
plt.figure(figsize=(15, 6))
off_season = [1, 2, 3, 11, 12]

try:
    print("🎨 Generating visualization from the already-trained LSTM...")
    
    # 2. PREDICT ONLY (We skip .fit() because it's already done!)
    # We use the variable 'best_model_lstm' directly from your previous block
    prediction_scaled = best_model_lstm.predict(n=len(y_val_scaled_ts), future_covariates=X_s_w_c_clean)
    
    # 3. Inverse Scale back to Kilograms
    raw_values = prediction_scaled.values().reshape(-1, 1)
    unscaled_values = y_scaler.inverse_transform(raw_values)
    prediction_kg = prediction_scaled.with_values(unscaled_values)
    
    # 4. Apply Physical Constraints
    prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
    pred_vals_final = prediction_kg.values().flatten()
    months = prediction_kg.time_index.month
    for j, month in enumerate(months):
        if month in off_season:
            pred_vals_final[j] = 0.0
            
    final_prediction = prediction_kg.with_values(pred_vals_final.reshape(-1, 1, 1))
    
    # 5. Plotting
    y_val_ts.plot(label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    final_prediction.plot(label='Tuned LSTM (V4) + Constraints', color='blueviolet', lw=2, linestyle='--')
    
    plt.title("Final Tuned LSTM Performance: Full Set (V4)", fontsize=16, fontweight='bold')
    plt.ylabel("Weight (kg)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    plt.show()

except Exception as e:
    print(f"❌ Error: {e}. (Make sure the cell with the random search has finished running!)")

## 5.5. TFT: Full Set V4

In [ ]:
# Perform hyperparameter tuning and evaluate results on validation set

# --- 1. Parameter Grid for TFT ---
# We focus on the number of attention heads and the hidden size
tft_param_grid = {
    "input_chunk_length": [14, 21],
    "output_chunk_length": [1],
    "hidden_size": [16, 32, 64],        # Size of the LSTM/Attention layers
    "lstm_layers": [1, 2],
    "num_attention_heads": [1, 4],      # How many "patterns" it can watch at once
    "dropout": [0.1, 0.2],
    "batch_size": [16, 32],
    "n_epochs": [100],                  # Consistent with your LSTM phase
    "add_relative_index": [True],       # Helps TFT understand time position
    "optimizer_kwargs": [{"lr": 1e-3}, {"lr": 5e-4}],
    "random_state": [42],
    "force_reset": [True],
    "pl_trainer_kwargs": [{"accelerator": "cpu"}]
}

print("🗼 Starting TFT Random Search on V4 (Full Set - Scaled)...")

# Perform Random Search
# n_random_samples=8 keeps the time manageable (TFT is slower than LSTM)
best_model_tft, best_params_tft, _ = TFTModel.gridsearch(
    parameters=tft_param_grid,
    series=y_train_scaled_ts,        # Training on SCALED data
    future_covariates=X_s_w_c_clean, # Using V4 Full Set
    val_series=y_val_scaled_ts,      # Validating on SCALED data
    metric=calculate_wape_with_constraints, 
    n_random_samples=20, 
    n_jobs=1,                        
    verbose=False
)

# --- 2. Re-fit and Final Evaluation ---
print(f"✅ Best TFT Params found: {best_params_tft}")

# Re-fit the best model
best_model_tft.fit(series=y_train_scaled_ts, future_covariates=X_s_w_c_clean)

# Predict (Result will be scaled)
prediction_scaled = best_model_tft.predict(n=len(y_val_scaled_ts), future_covariates=X_s_w_c_clean)

# --- 3. INVERSE SCALING (Rescale back to kg) ---
raw_values = prediction_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_kg = prediction_scaled.with_values(unscaled_values)

# --- 4. Apply Physical Constraints ---
# Enforce Non-Negativity
prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))

# Enforce Seasonal Zeroing (Jan, Feb, Mar, Nov, Dec)
off_season = [1, 2, 3, 11, 12]
pred_vals_kg = prediction_kg.values().flatten()
months = prediction_kg.time_index.month
for i, m in enumerate(months):
    if m in off_season:
        pred_vals_kg[i] = 0.0
final_pred_tft = prediction_kg.with_values(pred_vals_kg.reshape(-1, 1, 1))

# --- 5. Final Standardized Metrics ---
res_mae_tft = mae(y_val_ts, final_pred_tft)
res_rmse_tft = rmse(y_val_ts, final_pred_tft)
res_wape_tft = calculate_wape_with_constraints(y_val_ts, final_pred_tft)

tft_tuned_results = [{
    "Model": "TFT (Tuned)",
    "Subset": "Full Set (V4)",
    "MAE (kg)": round(res_mae_tft, 3),
    "RMSE (kg)": round(res_rmse_tft, 3),
    "WAPE": f"{res_wape_tft * 100:.2f}%"
}]

display(pd.DataFrame(tft_tuned_results))

In [ ]:
# Save the model so the long training of the model doesn't have to be repeated

best_model_tft.save("best_tft_v4_trained_20260301.pt")

In [ ]:
# Visualize the results

# 1. Setup the figure
plt.figure(figsize=(15, 6))
off_season = [1, 2, 3, 11, 12]

try:
    print("🎨 Generating visualization from the already-trained TFT...")
    
    # 2. PREDICT ONLY (No .fit() needed!)
    # Result will be scaled because the model was trained on y_train_scaled_ts
    prediction_scaled = best_model_tft.predict(n=len(y_val_scaled_ts), future_covariates=X_s_w_c_clean)
    
    # 3. Inverse Scale back to Kilograms
    raw_values = prediction_scaled.values().reshape(-1, 1)
    unscaled_values = y_scaler.inverse_transform(raw_values)
    prediction_kg = prediction_scaled.with_values(unscaled_values)
    
    # 4. Apply Physical Constraints
    # Constraint A: Non-Negativity
    prediction_kg = prediction_kg.map(lambda x: np.maximum(0, x))
    
    # Constraint B: Seasonal Zeroing
    pred_vals_final = prediction_kg.values().flatten()
    months = prediction_kg.time_index.month
    for j, month in enumerate(months):
        if month in off_season:
            pred_vals_final[j] = 0.0
            
    final_prediction = prediction_kg.with_values(pred_vals_final.reshape(-1, 1, 1))
    
    # 5. Plotting
    y_val_ts.plot(label='Actual (EX_WEIGHT)', color='black', lw=1.5)
    final_prediction.plot(label='Tuned TFT (V4) + Constraints', color='teal', lw=2, linestyle='--')
    
    plt.title("Final Tuned TFT Performance: Full Set (V4)", fontsize=16, fontweight='bold')
    plt.ylabel("Weight (kg)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    plt.show()

except Exception as e:
    print(f"❌ Error during visualization: {e}")

In [ ]:
# Define a function to use the same business logic as it was used during hyperparameter tuning

def apply_strawberry_constraints(pred_ts):
    """Ensures no negative sales and sets off-season (Nov-Mar) to zero."""
    # 1. Force Non-Negativity
    constrained_ts = pred_ts.map(lambda x: np.maximum(0, x))
    
    # 2. Seasonal Zeroing (Months 1, 2, 3, 11, 12)
    off_season = [1, 2, 3, 11, 12]
    values = constrained_ts.values().flatten()
    months = constrained_ts.time_index.month
    
    for i, m in enumerate(months):
        if m in off_season:
            values[i] = 0.0
            
    return constrained_ts.with_values(values.reshape(-1, 1, 1))

In [ ]:
# Retrain best model to include random state for reproducibility

best_params_tft["random_state"] = 42

# Re-initialize the model using the best params from your gridsearch
# This ensures it's a fresh object with no 'memory' of previous fits
best_model_tft = TFTModel(
    **best_params_tft,
)

print("🗼 Re-fitting TFT using the same logic as Gridsearch...")

# Fit using the FULL covariate series (X_s_w_c_clean) 
# Darts will slice this to match the training/validation series automatically.
best_model_tft.fit(
    series=y_train_scaled_ts, 
    future_covariates=X_s_w_c_clean, # Use the full series, not a subset
    val_series=y_val_scaled_ts,
    val_future_covariates=X_s_w_c_clean # Pass the same full series here
)

print("✅ Fit successful! Dimensions are now aligned.")

# --- Validation Prediction (Scaled) ---
# Use the end of the training set as context to predict the validation period
prediction_scaled = best_model_tft.predict(
    n=len(y_val_scaled_ts), 
    series=y_train_scaled_ts, 
    future_covariates=X_s_w_c_clean
)

# --- Post-Processing: Unscaling and Constraints ---
# Inverse Scaling
raw_values = prediction_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_kg = prediction_scaled.with_values(unscaled_values)

# Apply Physical Constraints (Function already defined in your environment)
final_pred_tft_val = apply_strawberry_constraints(prediction_kg)

# --- Updated Standardized Metrics ---
res_mae_tft = mae(y_val_ts, final_pred_tft_val)
res_rmse_tft = rmse(y_val_ts, final_pred_tft_val)
res_wape_tft = calculate_wape_with_constraints(y_val_ts, final_pred_tft_val)

tft_tuned_results = [{
    "Model": "TFT (Tuned - Deterministic)",
    "Subset": "Full Set (V4)",
    "MAE (kg)": round(res_mae_tft, 3),
    "RMSE (kg)": round(res_rmse_tft, 3),
    "WAPE": f"{res_wape_tft * 100:.2f}%"
}]

print("✅ TFT Re-training Complete.")
display(pd.DataFrame(tft_tuned_results))

## 5.6. Comparison of Tuned Models

As all selected models are hyperparameter-tuned now, let's look at the results and see which one performs the best.

In [ ]:
# Create table for comparison

# 1. Standardize the Naive variable to match the others
# We add the "Model" key and ensure "Subset" matches your table structure
naive_entry = [{
    "Model": "Naive Baseline",
    "Subset": "None",
    "MAE (kg)": naive_results[0]["MAE (kg)"],
    "RMSE (kg)": naive_results[0]["RMSE (kg)"],
    "WAPE": naive_results[0]["WAPE"]
}]

# 2. Combine all results into one master list
all_final_results = (
    naive_entry + 
    best_results +           # ARIMAX V4
    lgbm_tuned_results +     # LightGBM V3
    xgb_tuned_results +      # XGBoost V3
    lstm_tuned_results +     # LSTM V4
    tft_tuned_results        # TFT V4
)

# 3. Create the DataFrame
leaderboard_df = pd.DataFrame(all_final_results)

# 4. Sort by WAPE (converting string percentage to float for sorting)
leaderboard_df['WAPE_val'] = leaderboard_df['WAPE'].str.rstrip('%').astype(float)
leaderboard_df = leaderboard_df.sort_values(by="WAPE_val").drop(columns=['WAPE_val'])

# 5. Display the final results
print("🏆 FINAL PROJECT LEADERBOARD: Performance vs Baseline")
print("-" * 60)
display(leaderboard_df)

# Save for your final report
leaderboard_df.to_csv("strawberry_final_leaderboard.csv", index=False)

## 5.7. Stacking Ensemble Model

Now we will create a stacking ensemble model that uses all our five hyperparameter tuned models as base learners.

In [ ]:
# Creating the Stacking manually instead of using darts, so ARIMAX can be also included and the base learners can have different input dataframes

print("🚀 Generating manual meta-features with custom datasets per model...")

# 1. Generate predictions using the SPECIFIC dataset each model was trained on
# ARIMAX and Trees use the version WITH lags (e.g., X_s_w_c_joined)
# Deep Learning uses the version WITHOUT lags (X_s_w_c_clean)

p_arima = model_tuned.predict(n=len(y_val_ts), future_covariates=X_s_w_c_joined).values().flatten()
p_lgbm  = model_lgbm_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined).values().flatten()
p_xgb   = model_xgb_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined).values().flatten()

# Deep learning models stay on the 'clean' set
p_lstm  = best_model_lstm.predict(n=len(y_val_ts), future_covariates=X_s_w_c_clean).values().flatten()
p_tft   = best_model_tft.predict(n=len(y_val_ts), future_covariates=X_s_w_c_clean).values().flatten()

# 2. Standardize all to Scaled Space for the Meta-Learner
p_arima_scaled = y_scaler.transform(p_arima.reshape(-1, 1)).flatten()
# Note: p_lgbm and p_xgb should also be transformed if they were trained on raw kg
p_lgbm_scaled = y_scaler.transform(p_lgbm.reshape(-1, 1)).flatten()
p_xgb_scaled  = y_scaler.transform(p_xgb.reshape(-1, 1)).flatten()

# 3. Create Meta-Feature Matrix (X)
X_meta = np.column_stack([p_arima_scaled, p_lgbm_scaled, p_xgb_scaled, p_lstm, p_tft])
y_meta = y_val_scaled_ts.values().flatten()

# 4. Train Meta-Learner
meta_learner = LinearRegression(positive=True)
meta_learner.fit(X_meta, y_meta)

# 5. Build Final Forecast (Inverse Scaling & Constraints)
final_scaled_values = meta_learner.predict(X_meta)
prediction_stack_scaled = y_val_scaled_ts.with_values(final_scaled_values.reshape(-1, 1, 1))

raw_values = prediction_stack_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_stack_kg = prediction_stack_scaled.with_values(unscaled_values)

# 6. Apply Physical Constraints
# A. Non-Negativity
prediction_stack_kg = prediction_stack_kg.map(lambda x: np.maximum(0, x))

# B. Seasonal Zeroing (Jan, Feb, Mar, Nov, Dec)
off_season = [1, 2, 3, 11, 12]
pred_vals_stack = prediction_stack_kg.values().flatten()
months = prediction_stack_kg.time_index.month

for i, m in enumerate(months):
    if m in off_season:
        pred_vals_stack[i] = 0.0

final_pred_stack = prediction_stack_kg.with_values(pred_vals_stack.reshape(-1, 1, 1))

# 7. Metrics & Display
res_mae_stack = mae(y_val_ts, final_pred_stack)
res_rmse_stack = rmse(y_val_ts, final_pred_stack)
res_wape_stack = calculate_wape_with_constraints(y_val_ts, final_pred_stack)

stack_results = [{
    "Model": "Stacking Ensemble (Global 4)",
    "Subset": "Full Set (V4)",
    "MAE (kg)": round(res_mae_stack, 3),
    "RMSE (kg)": round(res_rmse_stack, 3),
    "WAPE": f"{res_wape_stack * 100:.2f}%"
}]

print("\n--- 🏆 Final Stacking Model Results ---")
display(pd.DataFrame(stack_results))

In [ ]:
# Visualizing the predictions on the timeline and also the weight the meta learner gave to the different base learners

# 1. Setup the figure
plt.figure(figsize=(16, 7))

# 2. Plot Actual Values
y_val_ts.plot(label='Actual Sales (EX_WEIGHT)', color='black', lw=1.5, alpha=0.6)

# 3. Plot the Stacking Ensemble Prediction
# Using a bold color like 'crimson' to highlight the final model
final_pred_stack.plot(label='Stacking Ensemble (Full 5 Models)', color='crimson', lw=2.5)

# 4. Add Visual Formatting
plt.title("Final Performance: Manual Stacking Ensemble vs. Actual Sales", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Weight (kg)", fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend(loc='upper left', frameon=True, shadow=True)

# 5. Save the result
plt.tight_layout()
plt.savefig("manual_stacking_ensemble_results.png", dpi=300)
plt.show()

# 6. Bonus: Visualize the "Model Weights"
# This bar chart shows exactly how much the Meta-Learner trusts each base model
model_names = ["ARIMAX", "LightGBM", "XGBoost", "LSTM", "TFT"]
weights = meta_learner.coef_

plt.figure(figsize=(10, 5))
plt.bar(model_names, weights, color=['gray', 'green', 'orange', 'blue', 'purple'], alpha=0.8)
plt.title("Meta-Learner: Importance Weights per Model", fontsize=14)
plt.ylabel("Weight Coefficient")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("meta_learner_weights.png")
plt.show()

### 5.7.1. Hyperparameter Tuning of Stacking Model

As with all the individual models, we will now also conduct hyperparameter tuning for the stacking model to optimize its perfromance.

In [ ]:
# Perform hyperparameter tuning and evaluate the results on the validation set

print("🚀 Generating manual meta-features...")

# 1. Generate predictions using the SPECIFIC dataset each model was trained on
p_arima = model_tuned.predict(n=len(y_val_ts), future_covariates=X_s_w_c_joined).values().flatten()
p_lgbm  = model_lgbm_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined).values().flatten()
p_xgb   = model_xgb_tuned.predict(n=len(y_val_ts), future_covariates=X_s_c_joined).values().flatten()
p_lstm  = best_model_lstm.predict(n=len(y_val_ts), future_covariates=X_s_w_c_clean).values().flatten()
p_tft   = best_model_tft.predict(n=len(y_val_ts), future_covariates=X_s_w_c_clean).values().flatten()

# 2. Standardize all to Scaled Space for the Meta-Learner
p_arima_scaled = y_scaler.transform(p_arima.reshape(-1, 1)).flatten()
p_lgbm_scaled = y_scaler.transform(p_lgbm.reshape(-1, 1)).flatten()
p_xgb_scaled  = y_scaler.transform(p_xgb.reshape(-1, 1)).flatten()

# 3. Create Meta-Feature Matrix (X) and Target (y)
X_meta = np.column_stack([p_arima_scaled, p_lgbm_scaled, p_xgb_scaled, p_lstm, p_tft])
y_meta = y_val_scaled_ts.values().flatten()

# --- 4. HYPERPARAMETER TUNING ON THE META-LEARNER ---
print("🔍 Tuning Meta-Learner (Ridge Regression)...")

# Define parameter grid for Alpha (regularization strength)
# We use positive=True to keep our physics constraints
param_grid = {'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
ridge = Ridge(positive=True, random_state=42)

# Use TimeSeriesSplit for cross-validation within the validation set
tscv = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    estimator=ridge, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='neg_mean_absolute_error'
)
grid_search.fit(X_meta, y_meta)

# Extract best meta-learner
meta_learner = grid_search.best_estimator_
print(f"✅ Best Meta-Learner Alpha: {grid_search.best_params_['alpha']}")

# 5. Build Final Forecast (Inverse Scaling & Constraints)
final_scaled_values = meta_learner.predict(X_meta)
prediction_stack_scaled = y_val_scaled_ts.with_values(final_scaled_values.reshape(-1, 1, 1))

raw_values = prediction_stack_scaled.values().reshape(-1, 1)
unscaled_values = y_scaler.inverse_transform(raw_values)
prediction_stack_kg = prediction_stack_scaled.with_values(unscaled_values)

# 6. Apply Physical Constraints
prediction_stack_kg = prediction_stack_kg.map(lambda x: np.maximum(0, x))

# Seasonal Zeroing
off_season = [1, 2, 3, 11, 12]
pred_vals_stack = prediction_stack_kg.values().flatten()
months = prediction_stack_kg.time_index.month
for i, m in enumerate(months):
    if m in off_season:
        pred_vals_stack[i] = 0.0

final_pred_stack = prediction_stack_kg.with_values(pred_vals_stack.reshape(-1, 1, 1))

# 7. Metrics & Display
res_mae_stack = mae(y_val_ts, final_pred_stack)
res_rmse_stack = rmse(y_val_ts, final_pred_stack)
res_wape_stack = calculate_wape_with_constraints(y_val_ts, final_pred_stack)

stack_results = [{
    "Model": "Stacked Ensemble (Tuned Meta)",
    "Best Alpha": grid_search.best_params_['alpha'],
    "MAE (kg)": round(res_mae_stack, 3),
    "RMSE (kg)": round(res_rmse_stack, 3),
    "WAPE": f"{res_wape_stack * 100:.2f}%"
}]

display(pd.DataFrame(stack_results))

In [ ]:
# Visualizing the predictions on the timeline and also the weight the meta learner gave to the different base learners

# 1. Setup the figure
plt.figure(figsize=(16, 7))

# 2. Plot Actual Values
y_val_ts.plot(label='Actual Sales (EX_WEIGHT)', color='black', lw=1.5, alpha=0.6)

# 3. Plot the Stacking Ensemble Prediction
# Using a bold color like 'crimson' to highlight the final model
final_pred_stack.plot(label='Stacking Ensemble (Full 5 Models)', color='crimson', lw=2.5)

# 4. Add Visual Formatting
plt.title("Final Performance: Manual Stacking Ensemble vs. Actual Sales", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Weight (kg)", fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend(loc='upper left', frameon=True, shadow=True)

# 5. Save the result
plt.tight_layout()
plt.savefig("manual_stacking_ensemble_results.png", dpi=300)
plt.show()

# 6. Bonus: Visualize the "Model Weights"
# This bar chart shows exactly how much the Meta-Learner trusts each base model
model_names = ["ARIMAX", "LightGBM", "XGBoost", "LSTM", "TFT"]
weights = meta_learner.coef_

plt.figure(figsize=(10, 5))
plt.bar(model_names, weights, color=['gray', 'green', 'orange', 'blue', 'purple'], alpha=0.8)
plt.title("Meta-Learner: Importance Weights per Model", fontsize=14)
plt.ylabel("Weight Coefficient")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("meta_learner_weights.png")
plt.show()

# 6. Model Evaluation on Test Set

As the training and tuning of all models is finalized at this stage, we will now evaluate the models' performance on the 2025 test set that none of the models has seen so far

In [ ]:
# Before evaluating the best models on the test set, let's compare the score of all of them on the validation set (now also including the stacking model)

# 1. Prepare the Stacking Entry to match the master format
# Note: These variables were calculated in your previous manual stacking block
stacking_entry = [{
    "Model": f"🏆 Stacking Ensemble (Tuned)",
    "Subset": "V3 & V4 (depending on base learner)",
    "MAE (kg)": round(res_mae_stack, 3),
    "RMSE (kg)": round(res_rmse_stack, 3),
    "WAPE": f"{res_wape_stack * 100:.2f}%"
}]

# 2. Re-standardize the Naive Baseline (ensuring consistency)
naive_entry = [{
    "Model": "Naive Baseline",
    "Subset": "None",
    "MAE (kg)": naive_results[0]["MAE (kg)"],
    "RMSE (kg)": naive_results[0]["RMSE (kg)"],
    "WAPE": naive_results[0]["WAPE"]
}]

# 3. Combine all results (Naive + 5 Individuals + Ensemble)
all_final_results = (
    naive_entry + 
    best_results +           # ARIMAX Tuned
    lgbm_tuned_results +     # LightGBM Tuned
    xgb_tuned_results +      # XGBoost Tuned
    lstm_tuned_results +     # LSTM Tuned
    tft_tuned_results +      # TFT Tuned
    stacking_entry           # The New Master Model
)

# 4. Create and Sort the DataFrame
leaderboard_df = pd.DataFrame(all_final_results)

# Convert WAPE string (e.g., "15.50%") to float for accurate sorting
leaderboard_df['WAPE_val'] = leaderboard_df['WAPE'].str.rstrip('%').astype(float)
leaderboard_df = leaderboard_df.sort_values(by="WAPE_val").drop(columns=['WAPE_val']).reset_index(drop=True)

# 5. Final Display
print("🏆 FINAL PROJECT LEADERBOARD: Performance vs Baseline")
print("-" * 75)
display(leaderboard_df)

# Save the comprehensive final version
leaderboard_df.to_csv("strawberry_final_leaderboard_complete.csv", index=False)

In [ ]:
# Create the 15-column 'Clean' set for LSTM & TFT. This ensures we don't feed manual lags into models that create their own internal lags

lag_components = ['lag_7', 'lag_14', 'rolling_mean_7']
X_test_s_w_c_clean = X_test_s_w_c_ts.drop_columns(lag_components)

print(f"✅ Clean Test Set created for Deep Learning. Components: {len(X_test_s_w_c_clean.components)}")
print(f"✅ Lagged Test Set ready for Trees/ARIMA. Components: {len(X_test_s_w_c_ts.components)}")

In [ ]:
# Prepare the covariates to perform evaluation on test set

# 1. Create a "Global" version of covariates for the ARIMAX predict call
# This merges Train, Val, and Test covariates into one continuous timeline
full_cov_V3_joined = concatenate([X_train_s_c_ts, X_val_s_c_ts, X_test_s_c_ts], axis=0)
full_cov_V4_joined = concatenate([X_train_s_w_c_ts, X_val_s_w_c_ts, X_test_s_w_c_ts], axis=0)
full_cov_V4_clean  = concatenate([X_s_w_c_clean, X_test_s_w_c_clean], axis=0)

print(f"✅ Full Covariates prepared. Range: {full_cov_joined.start_time()} to {full_cov_joined.end_time()}")

In [ ]:
# Perform the evaluation of all tuned models and rank them according to perfromance

test_results_list = []
all_final_predictions = {}
test_meta_features = [] 

# --- A. NAIVE BASELINE (Seasonal Persistence) ---
naive_values = y_val_ts.values()[-len(y_test_ts):]
naive_pred_kg = y_test_ts.with_values(naive_values)
naive_pred_kg = apply_strawberry_constraints(naive_pred_kg)

test_results_list.append({
    "Model": "Naive Baseline",
    "MAE (kg)": round(mae(y_test_ts, naive_pred_kg), 3),
    "RMSE (kg)": round(rmse(y_test_ts, naive_pred_kg), 3),
    "WAPE": calculate_wape_with_constraints(y_test_ts, naive_pred_kg)
})
all_final_predictions["Naive Baseline"] = naive_pred_kg

# Mapping models to their specific requirements
# Note: LightGBM and XGBoost 'y_is_scaled' changed to True if they were trained on scaled data
models_to_test = [
    {"model": model_tuned,      "cov": full_cov_V4_joined, "y_is_scaled": False, "name": "ARIMAX"},
    {"model": model_lgbm_tuned, "cov": full_cov_V3_joined, "y_is_scaled": False,  "name": "LightGBM"},
    {"model": model_xgb_tuned,  "cov": full_cov_V3_joined, "y_is_scaled": False,  "name": "XGBoost"},
    {"model": best_model_lstm,  "cov": full_cov_V4_clean,  "y_is_scaled": True,  "name": "LSTM"},
    {"model": best_model_tft,   "cov": full_cov_V4_clean,  "y_is_scaled": True,  "name": "TFT"}
]

for m_info in models_to_test:
    name = m_info["name"]
    
    # 1. Predict
    raw_pred = m_info["model"].predict(n=len(y_test_ts), series=y_val_ts, future_covariates=m_info["cov"])
    
    # 2. Scale Logic for Meta-Learner (Must be in 0-1 range)
    if m_info["y_is_scaled"]:
        test_meta_features.append(raw_pred.values().flatten())
        val_in_kg = y_scaler.inverse_transform(raw_pred.values().reshape(-1, 1))
        pred_in_kg_ts = raw_pred.with_values(val_in_kg)
    else:
        # For unscaled models (ARIMAX), scale it for the Meta-Learner
        val_scaled = y_scaler.transform(raw_pred.values().reshape(-1, 1))
        test_meta_features.append(val_scaled.flatten())
        pred_in_kg_ts = raw_pred

    # 3. Apply Physical Constraints
    final_pred_kg = apply_strawberry_constraints(pred_in_kg_ts)
    
    # SAVE FOR VISUALIZATION
    all_final_predictions[name] = final_pred_kg
    
    # --- ROBUST ALIGNMENT CHECK ---
    try:
        aligned_pred = y_test_ts.with_values(final_pred_kg.values()[:len(y_test_ts)])
        
        test_results_list.append({
            "Model": name,
            "MAE (kg)": round(mae(y_test_ts, aligned_pred), 3),
            "RMSE (kg)": round(rmse(y_test_ts, aligned_pred), 3),
            "WAPE": calculate_wape_with_constraints(y_test_ts, aligned_pred)
        })
    except Exception as e:
        print(f"⚠️ Error aligning {name}: {e}")
        continue

# --- 4. THE TUNED STACKING ENSEMBLE ---
try:
    X_meta_test = np.column_stack(test_meta_features)
    stack_scaled_vals = meta_learner.predict(X_meta_test)
    
    # Prevent scaling explosion: clip values to expected 0-1 range if needed
    stack_scaled_vals = np.clip(stack_scaled_vals, 0, 1)
    
    stack_unscaled_vals = y_scaler.inverse_transform(stack_scaled_vals.reshape(-1, 1))
    
    # Build TimeSeries using y_test_ts template
    final_stack_pred_kg = y_test_ts.with_values(stack_unscaled_vals.reshape(-1, 1, 1))
    final_stack_pred_kg = apply_strawberry_constraints(final_stack_pred_kg)
    
    # SAVE FOR VISUALIZATION
    all_final_predictions["Stacking Ensemble"] = final_stack_pred_kg

    test_results_list.append({
        "Model": "Stacking Ensemble",
        "MAE (kg)": round(mae(y_test_ts, final_stack_pred_kg), 3),
        "RMSE (kg)": round(rmse(y_test_ts, final_stack_pred_kg), 3),
        "WAPE": calculate_wape_with_constraints(y_test_ts, final_stack_pred_kg)
    })
except Exception as e:
    print(f"⚠️ Error in Stacking: {e}")

# Display Final Table
test_leaderboard = pd.DataFrame(test_results_list)
test_leaderboard['WAPE (%)'] = test_leaderboard['WAPE'].apply(lambda x: f"{x * 100:.2f}%")
display(test_leaderboard.sort_values(by="WAPE").drop(columns=['WAPE']).reset_index(drop=True))

In [ ]:
# Visualize the results

# 1. Setup the figure with 6 rows
fig, axes = plt.subplots(6, 1, figsize=(15, 35), sharex=False)

# 2. Define consistent colors
colors = ['gray', 'green', 'blue', 'orange', 'purple', 'brown', 'crimson']

# 3. Plotting Helper Function to ensure alignment
def plot_forecast(ax, name, color):
    # Plot Actuals (Gray background)
    y_test_ts.plot(ax=ax, label="Actual Sales", color="black", alpha=0.3, lw=1)
    
    # Plot the specific model from your saved dictionary
    if name in all_final_predictions:
        all_final_predictions[name].plot(ax=ax, label=f"{name} Forecast", color=color, lw=2)
    
    ax.set_title(f"Model Performance: {name}", fontsize=14, fontweight='bold')
    ax.set_ylabel("Kilograms")
    ax.tick_params(labelbottom=True) 
    ax.set_xlabel("Date (2025 Test Year)")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle="--", alpha=0.4)

# --- 4. EXECUTE MANUAL PLOTS ---

# Plot 1: ARIMAX
plot_forecast(axes[0], "ARIMAX", colors[1])

# Plot 2: LightGBM
plot_forecast(axes[1], "LightGBM", colors[2])

# Plot 3: XGBoost
plot_forecast(axes[2], "XGBoost", colors[3])

# Plot 4: LSTM
plot_forecast(axes[3], "LSTM", colors[4])

# Plot 5: TFT
plot_forecast(axes[4], "TFT", colors[5])

# Plot 6: Stacking Ensemble
plot_forecast(axes[5], "Stacking Ensemble", colors[6])

# Final touches
plt.xlabel("Date (2025 Test Year)")
plt.tight_layout()
plt.show()

# 7. Analysis of Deviations

Analyzing the deviations of the best performing model between the predictions and the actual values in order to get more in-depth insights on performance issues.

In [ ]:
# Join X_test and y_test to perfrom the analysis

# 1. Create base DataFrame with explicit DatetimeIndex
df_res = pd.DataFrame({
    'Actual': y_test_ts.values().flatten(),
    'Predicted': all_final_predictions['Stacking Ensemble'].values().flatten()
}, index=pd.to_datetime(y_test_ts.time_index))

# 2. FORCE X_test to have the same index before joining
# This is the "Safety Bridge" that ensures the join works
X_test_analysis = X_test.copy()
X_test_analysis.index = df_res.index 

# 3. Join the data
df_res = df_res.join(X_test_analysis)

# 4. Calculate Errors
df_res['Error'] = df_res['Actual'] - df_res['Predicted']
df_res['Abs_Error'] = df_res['Error'].abs()

# 5. Add Time Features
df_res['Day_of_Week'] = df_res.index.day_name()
df_res['Month'] = df_res.index.month
df_res['Is_Weekend'] = df_res.index.weekday >= 5

print(f"✅ Alignment Complete! Table Columns: {df_res.columns.tolist()}")
display(df_res)

## 7.1. General Analysis

This subchapter aims to derive some general insights about the forecasting errors, not focusing on one specific set of exogenous features

In [ ]:
# Get insights about which days were the top 20 with the highest MAE deviations

top_20_deviations = df_res.sort_values(by='Abs_Error', ascending=False).head(20)
display(top_20_deviations)

In [ ]:
# Analyzing underpredicition vs. overprediction

over_mask = df_res['Error'] < 0
under_mask = df_res['Error'] > 0
no_error_mask = df_res['Error'] == 0

# Count the days
count_over = over_mask.sum()
count_under = under_mask.sum()
count_no_error = no_error_mask.sum()

# Calculate Summed Absolute Error (Summed MAE) for each group
# We use the absolute value of errors for the overprediction group
sum_mae_over = df_res.loc[over_mask, 'Abs_Error'].sum()
sum_mae_under = df_res.loc[under_mask, 'Abs_Error'].sum()

# Display the results in a clean table
error_stats = pd.DataFrame({
    'Metric': ['Overpredictions', 'Underpredictions', 'No Error'],
    'Days Count': [count_over, count_under, count_no_error],
    'Summed MAE (kg)': [round(sum_mae_over, 3), round(sum_mae_under, 3), 0]
})

print("📊 Stacking Ensemble Error Distribution Summary:")
display(error_stats)

# 5. Visualizing the Balance
print(f"\n⚖️ Bias Check: The model is {'under-predicting' if sum_mae_under > sum_mae_over else 'over-predicting'} more volume in total.")

In [ ]:
# Binning of the actual sales amount in order to understand better which ranges of actual sales are not captured well by the model

# 1. Use the active season data (Apr-Oct)
df_active = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Create Deciles based on Actual Sales
df_active['Sales_Decile'] = pd.qcut(df_active['Actual'], 10, labels=False, duplicates='drop')

# 3. Calculate metrics per Decile
decile_analysis = df_active.groupby('Sales_Decile').agg({
    'Actual': ['min', 'max'],
    'Error': 'mean'
})
decile_analysis.columns = ['Min_Sales', 'Max_Sales', 'Mean_Bias']

# 4. Visualization
plt.figure(figsize=(14, 7))
colors = ['#ff4d4d' if x < 0 else '#4d4dff' for x in decile_analysis['Mean_Bias']]
bars = plt.bar(decile_analysis.index, decile_analysis['Mean_Bias'], color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

plt.axhline(0, color='black', linewidth=1.2)
plt.title("Model Bias by Sales Decile (Apr-Oct)", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("Mean Bias (Actual - Predicted) [kg]", fontsize=12)
plt.xlabel("Sales Decile (Actual Sales Ranges)", fontsize=12, labelpad=15)

# --- UPDATED LABELING: 1 decimal place ---
new_labels = [f"{row.Min_Sales:.1f}-{row.Max_Sales:.1f}kg" for i, row in decile_analysis.iterrows()]
plt.xticks(decile_analysis.index, new_labels, fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("📊 Decile Analysis Table (Rounded to 1 Decimal):")
display(decile_analysis.round(1))

In [ ]:
# Check the impact of the late start of the season in April

# 1. Identify the first day of sales in April 2025
# Filter for April and find the first row where Actual sales > 0
april_data = df_res[df_res.index.month == 4]
first_sale_date = april_data[april_data['Actual'] > 0].index.min()

# 2. Define the "False Start" period
# Starts April 1st and ends the day before the first recorded sale
false_start_start = "2025-04-01"
false_start_end = first_sale_date - pd.Timedelta(days=1)

# 3. Create the segments
df_false_start = df_res.loc[false_start_start : false_start_end]
df_active_season = df_res.loc[first_sale_date:] # From the first sale onwards

# 4. Impact Calculations
total_error_kg = df_res['Abs_Error'].sum()
false_start_error_kg = df_false_start['Abs_Error'].sum()
impact_on_total_error = (false_start_error_kg / total_error_kg) * 100

# Calculate WAPE for the active season only
# (Sum of Abs Error / Sum of Actuals)
active_wape = (df_active_season['Abs_Error'].sum() / df_active_season['Actual'].sum()) * 100

print(f"🔍 Season Detection:")
print(f"-> First recorded sale in April: {first_sale_date.date()}")
print(f"-> False Start Period: {false_start_start} to {false_start_end.date()}")

print(f"\n📉 Impact on 43% WAPE:")
print(f"-> Error from False Start: {false_start_error_kg:.2f} kg")
print(f"-> This period accounts for {impact_on_total_error:.2f}% of your total annual error.")
print(f"-> 'Clean' WAPE (Active Season Only): {active_wape:.2f}%")

## 7.2. Calendric-specific Analysis

This section focuses on exploring the effect of calendric features on the deviations.

In [ ]:
# Compare the error rates between different weekdays

# 1. Define the WAPE function
def calc_group_wape(group):
    if group['Actual'].sum() == 0:
        return 0
    return (group['Abs_Error'].sum() / group['Actual'].sum()) * 100

# 2. Group by Day_of_Week and calculate metrics
day_stats = df_res.groupby('Day_of_Week').agg({
    'Abs_Error': 'mean'
}).rename(columns={'Abs_Error': 'MAE (kg)'})

day_stats['RMSE (kg)'] = df_res.groupby('Day_of_Week').apply(
    lambda x: np.sqrt((x['Error']**2).mean())
)

day_stats['WAPE %'] = df_res.groupby('Day_of_Week').apply(calc_group_wape)

# --- NEW: SORTING LOGIC ---
# Define the correct chronological order
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Convert the index to a Categorical type with the defined order
day_stats.index = pd.CategoricalIndex(
    day_stats.index, 
    categories=weekday_order, 
    ordered=True
)

# Sort by the new categorical index instead of WAPE %
day_stats = day_stats.sort_index().round(3)
# ---------------------------

print("📊 Comprehensive Weekday Performance (Sorted Chronologically):")
display(day_stats)

In [ ]:
# Compare the error rates between different months

# 1. Filter for the core season (April to October)
df_seasonal = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Define a function to calculate all three metrics for a group
def calc_monthly_metrics(group):
    # MAE: Mean of Absolute Errors
    mae_val = group['Abs_Error'].mean()
    
    # RMSE: Square root of the mean of Squared Errors
    rmse_val = np.sqrt((group['Error']**2).mean())
    
    # WAPE: Sum of Abs Errors / Sum of Actuals * 100
    actual_sum = group['Actual'].sum()
    wape_val = (group['Abs_Error'].sum() / actual_sum * 100) if actual_sum != 0 else 0
    
    return pd.Series({
        'MAE (kg)': round(mae_val, 3),
        'RMSE (kg)': round(rmse_val, 3),
        'WAPE %': round(wape_val, 2)
    })

# 3. Apply the analysis
month_stats = df_seasonal.groupby('Month').apply(calc_monthly_metrics)

# 4. Map month numbers to names for readability
month_names = {4: 'April', 5: 'May', 6: 'June', 7: 'July', 
               8: 'August', 9: 'September', 10: 'October'}
month_stats.index = month_stats.index.map(month_names)

print("📅 Peak Season Performance Deep-Dive (MAE vs. RMSE vs. WAPE):")
display(month_stats)

In [ ]:
# Compare the error rates between public holidays pre-holiday days and regular days

# 1. Focus on Active Season
df_seasonal = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Create the "Holiday Eve" flag
# We shift the holiday flag -1 so that the day BEFORE a holiday is marked as 1
df_seasonal['holiday_eve_flag'] = df_seasonal['public_holiday_flag'].shift(-1).fillna(0)

# 3. Create a combined category column
def categorize_day(row):
    if row['public_holiday_flag'] == 1:
        return 'Public Holiday'
    elif row['holiday_eve_flag'] == 1:
        return 'Pre-Holiday (1 Day Before)'
    else:
        return 'Regular Day'

df_seasonal['Day_Type'] = df_seasonal.apply(categorize_day, axis=1)

# 4. Define the WAPE function
def calc_group_wape(group):
    actual_sum = group['Actual'].sum()
    if actual_sum == 0:
        return 0
    return (group['Abs_Error'].sum() / actual_sum) * 100

# 5. Group and Calculate Metrics
eve_analysis = df_seasonal.groupby('Day_Type').agg({
    'Abs_Error': 'mean',
    'Actual': 'mean'
}).rename(columns={'Abs_Error': 'MAE (kg)', 'Actual': 'Avg_Sales (kg)'})

# Calculate RMSE
eve_analysis['RMSE (kg)'] = df_seasonal.groupby('Day_Type').apply(
    lambda x: np.sqrt((x['Error']**2).mean())
)

# Calculate WAPE
eve_analysis['WAPE %'] = df_seasonal.groupby('Day_Type').apply(calc_group_wape)

# 6. Reorder index for logical flow
order = ['Regular Day', 'Pre-Holiday (1 Day Before)', 'Public Holiday']
eve_analysis = eve_analysis.reindex(order)

print("📅 Holiday 'Eve' Analysis - Active Season (Apr-Oct):")
display(eve_analysis.round(3))

## 7.3. Weather-specific Analysis

This section focuses on exploring the effect of meteorological features on the deviations.

In [ ]:
# Display correlations between meteorological features and the deviations

# 1. Filter for the Active Season (April to October)
df_active = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Define exact variable names from your screenshot
# (Including units as seen in image_4f9e23.png)
weather_vars = [
    'temperature_2m_mean (°C)', 
    'sunshine_duration (s)', 
    'precipitation_sum (mm)', 
    'cloud_cover_mean (%)', 
    'relative_humidity_2m_mean (%)', 
    'wind_gusts_10m_mean (km/h)',
    'precipitation_hours (h)',
    'pressure_msl_mean (hPa)'
]

# 3. Calculate correlation between weather and Absolute Error
# This shows which weather factors actually confuse the model during harvest
active_correlations = df_active[weather_vars].corrwith(df_active['Abs_Error']).sort_values(ascending=False)

# 4. Visualization
plt.figure(figsize=(12, 8))
# Red for variables that increase error, Blue for those that decrease it
colors = ['#d63031' if x > 0 else '#0984e3' for x in active_correlations]
active_correlations.plot(kind='barh', color=colors)

plt.title("Active Season (Apr-Oct): Weather Impact on Prediction Error", fontsize=14, fontweight='bold')
plt.xlabel("Correlation Coefficient with Abs_Error")
plt.axvline(0, color='black', lw=1)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Top 'Active Season' Error Drivers:")
print(active_correlations)

In [ ]:
# Analyze the weather patterns of the top 20 days with the highest MAE errors

# 1. Get the average weather for your Top 20 Errors
worst_20_weather = top_20_deviations[weather_vars].mean()

# 2. Get the average weather for the entire Active Season (Apr-Oct)
season_avg_weather = df_active[weather_vars].mean()

# 3. Create a comparison table
pattern_explorer = pd.DataFrame({
    'Worst 20 Days': worst_20_weather,
    'Season Average': season_avg_weather
})
pattern_explorer['Delta %'] = ((pattern_explorer['Worst 20 Days'] - pattern_explorer['Season Average']) / pattern_explorer['Season Average']) * 100

print("🧪 Pattern Explorer: How do the 'Worst Days' differ from a normal day?")
display(pattern_explorer.sort_values(by='Delta %', ascending=False))

## 7.4 Waste-centric Analysis

In [ ]:
# Analyzing consecutive overprediction streaks to understand when food waste occurs most

# 1. Use the active season data (Apr-Oct)
df_active = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Identify Overprediction days
df_active['is_over'] = df_active['Error'] < 0

# 3. Group contiguous blocks
df_active['streak_id'] = (df_active['is_over'] != df_active['is_over'].shift()).cumsum()

# 4. Aggregate streak data, including the time index
streak_stats = df_active[df_active['is_over'] == True].groupby('streak_id').agg({
    'is_over': 'count',
    'Abs_Error': 'sum',
    'Actual': 'sum'
}).rename(columns={'is_over': 'Duration_Days', 'Abs_Error': 'Total_Waste_kg'})

# 5. Add the Timeframe column (Start Date to End Date)
streak_stats['Timeframe'] = df_active[df_active['is_over'] == True].groupby('streak_id').apply(
    lambda x: f"{x.index.min().strftime('%Y-%m-%d')} to {x.index.max().strftime('%Y-%m-%d')}"
)

# 6. Filter: Only include streaks of 2 or more days
streak_analysis = streak_stats[streak_stats['Duration_Days'] >= 2].copy()

# 7. Final Formatting & Display
streak_analysis = streak_analysis[['Timeframe', 'Duration_Days', 'Total_Waste_kg', 'Actual']]
streak_analysis = streak_analysis.sort_values(by='Total_Waste_kg', ascending=False)

print(f"🚨 Found {len(streak_analysis)} overprediction streaks lasting 2+ days.")
display(streak_analysis)

In [ ]:
# Checking in which sequences a day of overprediction can be (partially) covered by a following day of underprediction, as strawberries can still be sold the following day

# 1. Focus on Active Season
df_active = df_res[(df_res.index.month >= 4) & (df_res.index.month <= 10)].copy()

# 2. Identify the Direction of Error
# Positive Error = Underprediction (Need more)
# Negative Error = Overprediction (Have extra)
df_active['Next_Day_Error'] = df_active['Error'].shift(-1)

# 3. Define the "Salvage Sequence"
# Sequence: Today is Over (Error < 0) AND Tomorrow is Under (Error > 0)
salvage_mask = (df_active['Error'] < 0) & (df_active['Next_Day_Error'] > 0)

# Identify the indices of these specific sequences
salvage_indices = df_active.index[salvage_mask]

# 4. Calculate "Salvage Adjustments"
# We create a copy of Absolute Error to modify it
df_active['Adjusted_Abs_Error'] = df_active['Abs_Error'].copy()

sequences_data = []

for idx in salvage_indices:
    today_over = abs(df_active.loc[idx, 'Error'])
    tomorrow_under = df_active.loc[idx, 'Next_Day_Error']
    
    # Logic: The leftover (Real Waste) is the difference
    # If Today Over 5 and Tomorrow Under 3 -> Real Waste is 2kg over 2 days.
    # If Today Over 2 and Tomorrow Under 5 -> Real Waste is 3kg under over 2 days.
    net_error = abs(today_over - tomorrow_under)
    
    # Store data for the report
    sequences_data.append({
        'Date': idx,
        'Today_Over': today_over,
        'Tomorrow_Under': tomorrow_under,
        'Net_Error': net_error,
        'Original_2Day_MAE': (today_over + tomorrow_under) / 2,
        'Adjusted_2Day_MAE': net_error / 2
    })
    
    # Apply the adjustment to the dataframe for the final MAE calculation
    # We replace the original Abs_Errors for those 2 days with the distributed net error
    df_active.loc[idx, 'Adjusted_Abs_Error'] = net_error / 2
    next_day = idx + pd.Timedelta(days=1)
    if next_day in df_active.index:
        df_active.loc[next_day, 'Adjusted_Abs_Error'] = net_error / 2

# 5. Final Metrics Comparison
original_sum_mae = df_active['Abs_Error'].sum()
adjusted_sum_mae = df_active['Adjusted_Abs_Error'].sum()

original_avg_mae = df_active['Abs_Error'].mean()
adjusted_avg_mae = df_active['Adjusted_Abs_Error'].mean()

print(f"🔄 Found {len(sequences_data)} Salvage Sequences (Over-then-Under)")
print("-" * 50)
print(f"📊 Summed MAE (Original): {original_sum_mae:.2f} kg")
print(f"📊 Summed MAE (Adjusted): {adjusted_sum_mae:.2f} kg")
print(f"📉 Improvement in Summed Error: {original_sum_mae - adjusted_sum_mae:.2f} kg")
print("-" * 50)
print(f"📈 Average MAE (Original): {original_avg_mae:.3f} kg")
print(f"📈 Average MAE (Adjusted): {adjusted_avg_mae:.3f} kg")

# Display first few sequences
display(pd.DataFrame(sequences_data).head(20))